# DR-VERGE — Streamlined Research Notebook

**Complementarity-Shift Distillation and INT8 deployment for lightweight two-field diabetic
retinopathy grading.**

Same experiment, same metrics, same figures and tables as the full pipeline — with the defensive
scaffolding removed and the data path made ~8× faster.

---

## Research questions

**RQ1 — Knowledge transfer.** To what extent does Complementarity-Shift Distillation preserve the
teacher's dual-view ordinal decision-shift structure and improve lightweight two-field DR grading,
relative to no distillation, logit distillation, and feature distillation?

Judged on two axes, so the answer is informative either way:
- *Predictive*: QWK (primary), Accuracy, Macro-F1, MAE, Severe-Error Rate
- *Mechanistic*: ShiftL1, cosine agreement, benefit correlation, dual-view gain

**RQ2 — Quantization.** To what extent can PTQ and QAT cut computational cost while preserving the
categorical and ordinal grading performance of the selected model?

---

## Locked protocol

| Item | Value |
|---|---|
| Primary metric | QWK (ordinal; grades 0<1<2<3<4) |
| Core seeds | 42, 123, 2026, 3407, 8888 |
| Tuning seeds | 42, 123, 2026 |
| Splits | DRTiD official; 800 train / 200 val / 550 test eyes |
| Selection | validation only, two-stage (method by mean QWK → checkpoint by best QWK) |
| Statistics | paired cluster bootstrap + permutation, B = P = 10,000, Holm per RQ family |
| Early stopping | every trainer, on validation QWK, `min_delta = 1e-4`; epoch budgets are ceilings |
| Quantization | eager backbone-only, identical scope for PTQ and QAT |
| External | DeepDRiD **Set-C** (confirmatory), Set-B/A supplementary |

**The comparison ladder.** Feature-KD and CSD each add ONE term to the same tuned logit-KD baseline:

```
no distillation → logit-KD → logit-KD + feature-KD → logit-KD + CSD
```

`alpha` and `tau` are tuned once for logit-KD and then **frozen**, so the only thing that changes at
the last step is the added term.

---

## What was simplified (and why it is safe)

| Removed | Why it is not needed |
|---|---|
| `PROTOCOL_HASH`, `RESUME_EXACT`, `completed` flags | Checkpoints are written **only when a job finishes**, so "the file exists" already means "the job completed". A fresh `RUN_TAG` gives a clean namespace. |
| Multi-session resume machinery | The image cache brings the run to ~5 h — one session. |
| PT2E supplementary path | It was disabled for the locked run anyway and never entered RQ2. |
| Pickled `model_object.pt` fallback | The quantized skeleton + `state_dict` reload is the supported path and is what the gate actually uses. |
| Local SSD staging, cache manifests, hardware provenance, pip-check gate | Operational plumbing; none of it affects a result. |

**The gates were NOT cut.** All 31 of them survive, because a gate is cheap and it is the thing that
catches a wrong result. What went away is the scaffolding *around* the science, not the checks on it:
the code is ~44% shorter (2,555 vs 4,524 lines) with the same 14 figures and the same tables.

**Nothing scientific was removed.** Same splits, same architectures, same CORAL construction, same
CSD formula, same seeds, same grids, same selection rule, same matched per-seed RQ2, same
statistics, same external protocol, same metrics, figures and tables.

## Efficiency

Three changes that cost nothing scientifically:

| Change | Effect |
|---|---|
| Resized-image cache (below) | decoding 32 ms → 0.2 ms per image; ~8× on the whole run |
| Teacher forward skipped when no distillation term uses it | the 10 single-view baselines and 5 no-distill runs stop paying for a discarded dual ResNet-50 forward |
| Dual-view gain in one forward pass instead of three | `forward()` already returns all three heads |
| Early stopping in **every** trainer (below) | epoch budgets become ceilings, not targets |

`USE_AMP` is available but **off by default** — mixed precision helps the ResNet-50 teacher, barely
helps the depthwise student, and changes numerics slightly.

## Early stopping

Every trainer — APTOS, teacher, student, QAT and the FP32 control — stops once validation QWK stops
improving, driven by one shared `EarlyStopping` object so the rule cannot drift between them. Epoch
counts in the config are **ceilings**, not targets, and each checkpoint records `epochs_run` and
`best_epoch` so the actual budget used is visible.

An epoch counts as an improvement only if it beats the best by more than `min_delta = 1e-4` QWK,
which is far below anything meaningful on 200 validation eyes — so patience is not reset by
numerical noise. Selection is unaffected: the best weights are kept and restored either way.

## What was made faster

Every transform starts with `Resize(224, 224)`, so the resized `uint8` image is cached in RAM after
its first decode. Verified **bit-identical** output: DRTiD decoding drops from 32 ms/image to
0.2 ms.

Together with the skipped teacher passes and early stopping everywhere, a full run is expected to
take roughly **5 hours on a T4** instead of ~46 h. That is an estimate from measured I/O timings and
an estimated GPU cost, not a measured end-to-end figure.

## 1 — Environment

In [ ]:
# Set before torch initialises CUDA -- required for deterministic cuBLAS kernels.
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Colab already ships a CUDA-enabled torch; only the extras are installed.
!pip install -q "albumentations==1.4.21" "scikit-learn>=1.6" "pandas>=2.2" "tqdm>=4.67"                "psutil>=6.0" "scipy>=1.14" "openpyxl>=3.1" "onnx>=1.17" "onnxruntime>=1.19"
!pip install -q "onnxscript>=0.1.0" || echo "onnxscript unavailable -- ONNX export will be reported as failed"

import torch, torchvision, sklearn, platform
print("torch      :", torch.__version__)
print("torchvision:", torchvision.__version__)
print("sklearn    :", sklearn.__version__)
print("python     :", platform.python_version())
print("GPU        :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
print("quant engines:", torch.backends.quantized.supported_engines)

## 2 — Configuration

Everything that defines the experiment is here. `RUN_TAG` isolates this run's artifacts — use a new
tag for a new run rather than writing into an old folder.

In [ ]:
import json, math, random, time, copy, glob, hashlib
import numpy as np, pandas as pd
import torch.nn as nn, torch.nn.functional as F

# ---------------- EDIT THESE ----------------
DRIVE_BASE = "/content/drive/MyDrive/DR-VERGE"
RUN_TAG    = "simple_v1"
QUICK      = False        # True = tiny rehearsal (1 seed, few epochs) to prove the pipeline runs
# Mixed precision. OFF by default: it is faster on the ResNet-50 teacher but changes numerics
# slightly, and the depthwise student barely benefits. Turn it on only if you accept that the
# numbers will differ marginally from an FP32 run.
USE_AMP    = False
# --------------------------------------------

DATASET_ROOT = f"{DRIVE_BASE}/dataset"

def _find_drtid(root):
    for c in (f"{root}/DRTiD/DRTiD", f"{root}/DRTiD"):
        if os.path.exists(f"{c}/Ground Truths/DR_grade/a. DR_grade_Training.csv"):
            return c
    raise FileNotFoundError("DRTiD not found under " + root)

def _find_deepdrid(root):
    for c in (f"{root}/DeepDRiD-master/regular_fundus_images", f"{root}/DeepDRiD/regular_fundus_images",
              f"{root}/DeepDRiD-master", f"{root}/DeepDRiD"):
        if os.path.exists(f"{c}/regular-fundus-validation/regular-fundus-validation.csv"):
            return c
    return None

DRTID_ROOT       = _find_drtid(DATASET_ROOT)
DRTID_IMAGE_ROOT = f"{DRTID_ROOT}/Original Images"
APTOS_ROOT       = f"{DATASET_ROOT}/APTOS"
DEEPDRID_ROOT    = _find_deepdrid(DATASET_ROOT)

ART = f"{DRIVE_BASE}/artifacts_{RUN_TAG}"
CKPT_DIR, MODELS_DIR = f"{ART}/checkpoints", f"{ART}/models"
RESULTS_DIR = f"{ART}/results"
FIG_DIR, TAB_DIR = f"{RESULTS_DIR}/figures", f"{RESULTS_DIR}/tables"
MET_DIR, PRED_DIR, LOG_DIR = f"{RESULTS_DIR}/metrics", f"{RESULTS_DIR}/predictions", f"{RESULTS_DIR}/logs"
CFG_DIR, SPLIT_DIR = f"{ART}/configs", f"{ART}/splits"
for d in (ART, CKPT_DIR, MODELS_DIR, RESULTS_DIR, FIG_DIR, TAB_DIR, MET_DIR, PRED_DIR,
          LOG_DIR, CFG_DIR, SPLIT_DIR):
    os.makedirs(d, exist_ok=True)

# ================= LOCKED PROTOCOL =================
SEEDS_CORE   = [42, 123, 2026, 3407, 8888]   # every core condition and every RQ2 variant
SEEDS_TUNING = [42, 123, 2026]               # hyperparameters scored as the MEAN over these
SEEDS_ABL    = [42, 123, 2026]               # formulation ablations
PRIMARY_SEED = 42

IMG_SIZE, NUM_CLASSES = 224, 5
NUM_THRESHOLDS = NUM_CLASSES - 1
SPLIT_SEED, VAL_FRACTION = 42, 0.20
POS_WEIGHT_MODE  = "sqrt"                              # pos_weight = sqrt(N_neg / N_pos)
STUDENT_CHANNELS = (32, 64, 96, 128, 160, 192, 224)    # ~330K-param student

# Every budget is a CEILING; training stops early once validation QWK stops improving.
# `min_delta` is the improvement that counts as real -- 1e-4 QWK is below anything meaningful on
# 200 validation eyes, so it only stops the patience counter being reset by numerical noise.
EARLY_STOP_MIN_DELTA = 1e-4
APTOS_R50_CFG   = {"epochs": 20, "lr": 1e-4, "batch_size": 32, "patience": 5}
APTOS_LIGHT_CFG = {"epochs": 30, "lr": 1e-3, "batch_size": 32, "patience": 6}
TEACHER_CFG = {"freeze_epochs": 5, "finetune_epochs": 20, "freeze_lr": 3e-4, "finetune_lr": 1e-5,
               "batch_size": 16, "patience": 8, "lambda_aux": 0.3}
STUDENT_CFG = {"epochs": 40, "lr": 1e-3, "batch_size": 16, "patience": 8, "lambda_aux": 0.5}
QAT_CFG     = {"epochs": 10, "patience": 4, "batch_size": 16, "lr_grid": [1e-5, 3e-5, 1e-4]}

# Pre-registered grids. CSD does NOT re-tune alpha/tau -- both are inherited from the logit-KD
# winner, so "logit-KD" and "logit-KD + CSD" differ by exactly one term.
GRID_LOGITKD = [{"alpha": 0.25, "tau_kd": 2.0}, {"alpha": 0.5, "tau_kd": 2.0},
                {"alpha": 0.5,  "tau_kd": 4.0}, {"alpha": 1.0, "tau_kd": 2.0}]
GRID_FEATKD  = [{"gamma_feat": 0.1}, {"gamma_feat": 0.5}, {"gamma_feat": 1.0}, {"gamma_feat": 2.0}]
GRID_CSD     = [{"csd_variant": "smoothl1_norm", "beta": 0.1},
                {"csd_variant": "smoothl1_norm", "beta": 0.2},
                {"csd_variant": "smoothl1_norm", "beta": 0.5},
                {"csd_variant": "magnitude_weighted_direction", "beta": 0.2}]

SELECTION_TIE_EPS = 0.005
BOOTSTRAP_B = PERMUTATION_P = 10000
ALPHA_CI = 0.05
CORE_CONDITIONS = ["dual_no_distill", "dual_logitkd", "dual_featkd", "dual_csd"]
PREREGISTERED = {
    "RQ1": [("dual_csd", "dual_no_distill"), ("dual_csd", "dual_logitkd"), ("dual_csd", "dual_featkd")],
    "RQ2": [("ptq_int8", "best_fp32"), ("qat_int8", "best_fp32"), ("qat_int8", "ptq_int8"),
            ("qat_int8", "fp32_ft_control"), ("qat_int8", "ft_ptq_int8")],
}
DEPLOY_RULE = {"min_qwk_retention_pct": 95.0,          # pre-specified ENGINEERING criterion,
               "note": "engineering retention criterion, not a clinical margin",
               "severe_error_must_not_credibly_worsen": True,
               "tiebreak": "lowest median CPU latency", "fallback": "best_fp32"}
BENCH = {"batch_size": 1, "warmup": 50, "runs": 500, "threads": 1, "repeats": 5}
DEEPDRID_PRIMARY_SUBSET = "setC"                        # the challenge's final-evaluation partition
DEEPDRID_PRIMARY_ORDER  = "_1=macula"                   # pre-registered; reverse is a sensitivity check
DEEPDRID_ORDERS = [DEEPDRID_PRIMARY_ORDER, "_1=disc"]

# Quantization backend, resolved once so PTQ and QAT can never differ.
_eng = list(torch.backends.quantized.supported_engines)
QUANT_ENGINE = "x86" if "x86" in _eng else ("fbgemm" if "fbgemm" in _eng else _eng[0])
torch.backends.quantized.engine = QUANT_ENGINE
QUANT_SCOPE = "eager_backbone_only"

if QUICK:                       # rehearsal: prove every stage runs; says NOTHING about accuracy
    SEEDS_CORE, SEEDS_TUNING, SEEDS_ABL = [42], [42], [42]
    APTOS_R50_CFG.update(epochs=1, patience=1); APTOS_LIGHT_CFG.update(epochs=1, patience=1)
    TEACHER_CFG.update(freeze_epochs=1, finetune_epochs=1, patience=1)
    STUDENT_CFG.update(epochs=2, patience=2)
    QAT_CFG.update(epochs=1, patience=1, lr_grid=[3e-5])
    GRID_LOGITKD, GRID_FEATKD, GRID_CSD = GRID_LOGITKD[:2], GRID_FEATKD[:2], GRID_CSD[:2]
    BOOTSTRAP_B = PERMUTATION_P = 300
    BENCH = {"batch_size": 1, "warmup": 3, "runs": 10, "threads": 1, "repeats": 2}

CONFIG = dict(run_tag=RUN_TAG, quick=QUICK, seeds_core=SEEDS_CORE, seeds_tuning=SEEDS_TUNING,
              seeds_ablation=SEEDS_ABL, split_seed=SPLIT_SEED, val_fraction=VAL_FRACTION,
              img_size=IMG_SIZE, num_classes=NUM_CLASSES, pos_weight_mode=POS_WEIGHT_MODE,
              student_channels=list(STUDENT_CHANNELS),
              early_stop_min_delta=EARLY_STOP_MIN_DELTA, use_amp=USE_AMP,
              aptos_r50=APTOS_R50_CFG,
              aptos_light=APTOS_LIGHT_CFG, teacher=TEACHER_CFG, student=STUDENT_CFG, qat=QAT_CFG,
              grid_logitkd=GRID_LOGITKD, grid_featkd=GRID_FEATKD, grid_csd=GRID_CSD,
              bootstrap_B=BOOTSTRAP_B, permutation_P=PERMUTATION_P,
              quant_engine=QUANT_ENGINE, quant_scope=QUANT_SCOPE, deploy_rule=DEPLOY_RULE,
              deepdrid_primary=DEEPDRID_PRIMARY_SUBSET, deepdrid_order=DEEPDRID_PRIMARY_ORDER)

print("DRTiD    :", DRTID_ROOT)
print("APTOS    :", APTOS_ROOT)
print("DeepDRiD :", DEEPDRID_ROOT or "NOT FOUND -- external validation will be skipped")
print("artifacts:", ART)
print("mode     :", "QUICK rehearsal" if QUICK else "FULL RUN", "| quant engine:", QUANT_ENGINE)

## 3 — Utilities & gates

A **gate** is a check that can stop the run. Anything computed after a failed upstream check is not
interpretable, so blocking gates raise instead of warning. The report is written on every call, so
it survives the abort.

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed, deterministic=True):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        try:
            torch.use_deterministic_algorithms(True, warn_only=True)
        except Exception as e:
            print("  deterministic algorithms unavailable:", e)

class EarlyStopping:
    """Tracks the best validation score, keeps its weights, and says when to stop.

    Used by all four trainers so the rule cannot drift between them: an epoch counts as an
    improvement only if it beats the best by more than `min_delta`, otherwise patience decrements.
    The best weights are kept in memory and written once at the end, which is also why a checkpoint
    file existing means the job finished.
    """
    def __init__(self, patience, min_delta=None, mode="max"):
        self.patience = patience
        self.min_delta = EARLY_STOP_MIN_DELTA if min_delta is None else min_delta
        self.best, self.best_state, self.bad, self.best_epoch = -np.inf, None, 0, -1

    def step(self, score, model, epoch):
        """Returns True when training should stop."""
        if np.isfinite(score) and score > self.best + self.min_delta:
            self.best, self.bad, self.best_epoch = float(score), 0, int(epoch)
            self.best_state = copy.deepcopy(model.state_dict())
            return False
        self.bad += 1
        if self.bad >= self.patience:
            print(f"  early stop @ep{epoch} (best {self.best:.4f} @ep{self.best_epoch}, "
                  f"no gain > {self.min_delta:g} for {self.patience} epochs)")
            return True
        return False

    def restore(self, model):
        if self.best_state is not None: model.load_state_dict(self.best_state)
        return model

def save_json(obj, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f: json.dump(obj, f, indent=2, default=str)

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()

def file_size_mb(p): return os.path.getsize(p) / 1024 ** 2 if p and os.path.exists(p) else float("nan")

def torch_save(obj, path, retries=4):
    """Atomic save with a short retry -- Drive's FUSE mount occasionally rejects a write."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    for i in range(retries):
        try:
            torch.save(obj, path + ".tmp"); os.replace(path + ".tmp", path); return path
        except (RuntimeError, OSError) as e:
            if i == retries - 1: raise
            print(f"  save retry {i+1}: {e}"); time.sleep(1.5 * (i + 1))

def torch_load(path, map_location="cpu", retries=4):
    for i in range(retries):
        try:
            return torch.load(path, map_location=map_location, weights_only=False)
        except (FileNotFoundError, OSError) as e:
            if i == retries - 1: raise
            print(f"  load retry {i+1}: {e}"); time.sleep(1.5 * (i + 1))

def load_weights(path, map_location="cpu"):
    """Checkpoints are {'model_state': ..., ...}; deployment artifacts are a bare state_dict."""
    raw = torch_load(path, map_location)
    return raw["model_state"] if isinstance(raw, dict) and "model_state" in raw else raw

# ---- gates ----
GATES = {}
class GateFailure(RuntimeError): pass

def gate(name, passed, detail="", blocking=False):
    GATES[name] = {"passed": bool(passed), "blocking": bool(blocking), "detail": detail}
    print(f"{'PASS' if passed else 'FAIL'} | {name}" + (f" | {detail}" if detail else ""))
    pd.DataFrame([{"gate": k, **v} for k, v in GATES.items()]).to_csv(
        f"{TAB_DIR}/table_gate_report.csv", index=False)
    if blocking and not passed:
        raise GateFailure(f"{name} failed: {detail}")
    return passed

BLOCK = not QUICK      # convergence/completeness gates only block the real run

_env_ok = torch.cuda.is_available() and bool({"x86", "fbgemm", "onednn"} & set(_eng))
gate("Gate0_Environment", _env_ok,
     f"GPU={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}, "
     f"quant engine={QUANT_ENGINE}", blocking=BLOCK)
save_json(CONFIG, f"{CFG_DIR}/config.json")
set_seed(PRIMARY_SEED)
print("device:", DEVICE)

## 4 — DRTiD splits (Gate 1)

DRTiD ships an official train/test split, used as-is; only train/val is carved out of the official
training rows, **stratified by grade** because Grade 4 is ~3.9% of them.

`Macula` and `Optic disc` columns are the source of truth for pairing. DRTiD's public metadata
exposes only a per-eye `ID` — no usable patient key — so clustering is **eye/record level** and is
never described as patient-level.

In [ ]:
from sklearn.model_selection import train_test_split

DRTID_EXPECTED = {"train": 800, "val": 200, "test": 550}

def make_splits():
    out = {k: f"{SPLIT_DIR}/drtid_{k}.csv" for k in ("train", "val", "test")}
    tr_csv = f"{DRTID_ROOT}/Ground Truths/DR_grade/a. DR_grade_Training.csv"
    te_csv = f"{DRTID_ROOT}/Ground Truths/DR_grade/b. DR_grade_Testing.csv"
    off_tr, off_te = pd.read_csv(tr_csv), pd.read_csv(te_csv)
    problems = []

    for nm, df in (("train", off_tr), ("test", off_te)):
        miss = [c for c in ("ID", "Grade", "Macula", "Optic disc", "LR") if c not in df.columns]
        if miss: problems.append(f"official {nm}: missing columns {miss}")
        elif df[["ID", "Grade", "Macula", "Optic disc", "LR"]].isnull().any().any():
            problems.append(f"official {nm}: null values in required columns")
        if "ID" in df and df["ID"].duplicated().any(): problems.append(f"official {nm}: duplicate IDs")
        if "Grade" in df:
            bad = sorted(set(df["Grade"].dropna().unique()) - set(range(NUM_CLASSES)))
            if bad: problems.append(f"official {nm}: grades outside 0-4 {bad}")
    if set(off_tr["ID"]) & set(off_te["ID"]): problems.append("official train/test share IDs")
    # one image must belong to exactly one eye-record
    imgs = pd.concat([off_tr["Macula"], off_tr["Optic disc"], off_te["Macula"], off_te["Optic disc"]]).astype(str)
    if imgs.duplicated().any(): problems.append("an image is assigned to more than one record")

    def std(df):
        # Filenames, not absolute paths: the split is data identity, not a machine path.
        return pd.DataFrame({"record_id": df["ID"],
                             "macula_filename": df["Macula"].astype(str) + ".jpg",
                             "disc_filename":   df["Optic disc"].astype(str) + ".jpg",
                             "grade": df["Grade"], "laterality": df["LR"]})

    if not all(os.path.exists(p) for p in out.values()):
        tr_ids, va_ids = train_test_split(off_tr["ID"].values, test_size=VAL_FRACTION,
                                          random_state=SPLIT_SEED, stratify=off_tr["Grade"].values)
        std(off_tr[off_tr["ID"].isin(tr_ids)]).to_csv(out["train"], index=False)
        std(off_tr[off_tr["ID"].isin(va_ids)]).to_csv(out["val"], index=False)
        std(off_te).to_csv(out["test"], index=False)
    else:
        print("Splits already exist -- reusing them (identical splits across sessions).")

    dfs = {k: pd.read_csv(v) for k, v in out.items()}
    for a, b in (("train", "val"), ("val", "test"), ("train", "test")):
        if set(dfs[a].record_id) & set(dfs[b].record_id): problems.append(f"{a}/{b} record overlap")

    rows = []
    for k, df in dfs.items():
        if len(df) != DRTID_EXPECTED[k]:
            problems.append(f"{k}: {len(df)} eyes, expected {DRTID_EXPECTED[k]}")
        missing = [f for c in ("macula_filename", "disc_filename") for f in df[c]
                   if not os.path.exists(f"{DRTID_IMAGE_ROOT}/{f}")]
        if missing: problems.append(f"{k}: {len(missing)} images missing, e.g. {missing[:2]}")
        d = df["grade"].value_counts().sort_index()
        if set(range(NUM_CLASSES)) - set(d.index): problems.append(f"{k}: not all grades present")
        rows.append({"split": k, "eyes": len(df), "images": 2 * len(df),
                     **{f"grade_{g}": int(d.get(g, 0)) for g in range(NUM_CLASSES)}})
    stats = pd.DataFrame(rows)
    stats.to_csv(f"{TAB_DIR}/table_00_dataset_statistics.csv", index=False)
    print(stats.to_string(index=False))
    save_json({k: {"file": os.path.basename(v), "sha256": sha256_file(v)} for k, v in out.items()},
              f"{CFG_DIR}/split_manifest.json")
    gate("Gate1_DRTiD", not problems,
         f"800/200/550 eyes; schema, uniqueness, image existence and split disjointness verified"
         if not problems else " | ".join(problems[:4]), blocking=BLOCK)
    return out["train"], out["val"], out["test"]

TRAIN_CSV, VAL_CSV, TEST_CSV = make_splits()

## 5 — APTOS integrity (Gate 1b)

In [ ]:
def check_aptos():
    problems, ids, rows = [], {}, []
    for name, csv, imgs in (("train", f"{APTOS_ROOT}/train_1.csv", f"{APTOS_ROOT}/train_images/train_images"),
                            ("val",   f"{APTOS_ROOT}/valid.csv",   f"{APTOS_ROOT}/val_images/val_images")):
        if not os.path.exists(csv): problems.append(f"{name}: CSV missing"); continue
        df = pd.read_csv(csv)
        if df["id_code"].duplicated().any(): problems.append(f"{name}: duplicate id_code")
        bad = sorted(set(df["diagnosis"].dropna().unique()) - set(range(NUM_CLASSES)))
        if bad: problems.append(f"{name}: diagnosis outside 0-4 {bad}")
        miss = [i for i in df["id_code"] if not os.path.exists(f"{imgs}/{i}.png")]
        if miss: problems.append(f"{name}: {len(miss)} images missing")
        ids[name] = set(df["id_code"]); rows.append({"split": name, "n": len(df),
                                                     "sha256": sha256_file(csv)})
    if len(ids) == 2 and (ids["train"] & ids["val"]):
        problems.append(f"train/val share {len(ids['train'] & ids['val'])} id_code(s)")
    save_json(rows, f"{CFG_DIR}/aptos_manifest.json")
    gate("Gate1b_APTOS", not problems,
         f"train={rows[0]['n']} val={rows[1]['n']}; ids unique and disjoint; all images resolve"
         if not problems else " | ".join(problems[:4]), blocking=BLOCK)

check_aptos()

## 6 — Image cache, transforms, datasets

**The one real optimisation.** Every transform begins with `Resize(224, 224)`, so the resized
`uint8` array is cached after the first decode and reused for every later epoch. Augmentation still
runs per sample on the cached array, so the result is **bit-identical** — only the repeated JPEG
decoding disappears (32 ms → 0.2 ms per image).

Horizontal flip is deliberately omitted: it would change macula/disc laterality semantics, and the
CrossFiT reference loader omits it for the same reason. Geometry is shared across the two fields of
one eye (they share an acquisition geometry); photometric jitter stays independent.

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

DRTID_MEAN, DRTID_STD = [0.372487, 0.217266, 0.119367], [0.281526, 0.179457, 0.109162]
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

_RESIZE = A.Resize(IMG_SIZE, IMG_SIZE)
IMAGE_CACHE = {}

def load_img(path):
    """Decode once, keep the resized uint8 array. Identical to decoding every time, because
    Resize(224) is the first op of every transform."""
    a = IMAGE_CACHE.get(path)
    if a is None:
        a = _RESIZE(image=np.array(Image.open(path).convert("RGB")))["image"]
        IMAGE_CACHE[path] = a
    return a

def prefill_cache(paths, label):
    todo = [p for p in dict.fromkeys(paths) if p not in IMAGE_CACHE]
    for p in tqdm(todo, desc=f"caching {label}", leave=False):
        load_img(p)
    mb = sum(a.nbytes for a in IMAGE_CACHE.values()) / 1024 ** 2
    print(f"  {label}: {len(todo)} decoded, cache now {len(IMAGE_CACHE)} images ({mb:.0f} MB)")

def build_tf(train, mean, std):
    ops = [A.Resize(IMG_SIZE, IMG_SIZE)]
    if train: ops += [A.Rotate(limit=15, p=0.7),
                      A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5)]
    return A.Compose(ops + [A.Normalize(mean=mean, std=std), ToTensorV2()])

class PairedTransform:
    """Same geometry on both fields (replayed), independent photometric jitter."""
    def __init__(self, mean, std):
        self.geo = A.ReplayCompose([A.Resize(IMG_SIZE, IMG_SIZE), A.Rotate(limit=15, p=0.7)])
        self.photo = A.Compose([A.RandomBrightnessContrast(0.15, 0.15, p=0.5),
                                A.Normalize(mean=mean, std=std), ToTensorV2()])
    def __call__(self, a, b):
        g = self.geo(image=a)
        b2 = A.ReplayCompose.replay(g["replay"], image=b)["image"]
        return self.photo(image=g["image"])["image"], self.photo(image=b2)["image"]

train_tf  = build_tf(True,  DRTID_MEAN, DRTID_STD)
eval_tf   = build_tf(False, DRTID_MEAN, DRTID_STD)
paired_tf = PairedTransform(DRTID_MEAN, DRTID_STD)
aptos_train_tf = build_tf(True,  IMAGENET_MEAN, IMAGENET_STD)
aptos_eval_tf  = build_tf(False, IMAGENET_MEAN, IMAGENET_STD)

PREPROCESSING = {"input_size": [IMG_SIZE, IMG_SIZE], "normalization_mean": DRTID_MEAN,
                 "normalization_std": DRTID_STD, "horizontal_flip": False,
                 "views": ["macula", "optic_disc"], "decision_threshold": 0.5}

class DualViewDataset(Dataset):
    """One row per EYE. `cluster_id` is the resampling unit for the bootstrap."""
    def __init__(self, csv, transform=None, paired=None, image_root=None):
        self.df = pd.read_csv(csv)
        root = image_root or DRTID_IMAGE_ROOT
        self.df["macula_path"] = self.df["macula_filename"].apply(lambda f: f"{root}/{f}")
        self.df["disc_path"]   = self.df["disc_filename"].apply(lambda f: f"{root}/{f}")
        self.transform, self.paired = transform or eval_tf, paired
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        m, d = load_img(r["macula_path"]), load_img(r["disc_path"])
        if self.paired is not None:
            a, b = self.paired(m, d)
        else:
            a, b = self.transform(image=m)["image"], self.transform(image=d)["image"]
        return {"macula": a, "disc": b, "label": torch.tensor(int(r["grade"]), dtype=torch.long),
                "cluster_id": int(r["record_id"])}

class APTOSDataset(Dataset):
    def __init__(self, csv, root, transform):
        self.df, self.root, self.tf = pd.read_csv(csv), root, transform
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        return {"image": self.tf(image=load_img(f"{self.root}/{r['id_code']}.png"))["image"],
                "label": torch.tensor(int(r["diagnosis"]), dtype=torch.long)}

def loader(ds, bs, shuffle, seed=None):
    # num_workers=0: the cache makes loading trivial, and single-process avoids the worker
    # teardown races that fill the log with exceptions.
    g = None
    if seed is not None:
        g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=0, generator=g)

# Warm the cache once so every later epoch is pure GPU work.
_drtid_paths = []
for csv in (TRAIN_CSV, VAL_CSV, TEST_CSV):
    d = pd.read_csv(csv)
    _drtid_paths += [f"{DRTID_IMAGE_ROOT}/{f}" for f in d["macula_filename"]]
    _drtid_paths += [f"{DRTID_IMAGE_ROOT}/{f}" for f in d["disc_filename"]]
prefill_cache(_drtid_paths, "DRTiD")
prefill_cache([f"{APTOS_ROOT}/train_images/train_images/{i}.png"
               for i in pd.read_csv(f"{APTOS_ROOT}/train_1.csv")["id_code"]], "APTOS train")
prefill_cache([f"{APTOS_ROOT}/val_images/val_images/{i}.png"
               for i in pd.read_csv(f"{APTOS_ROOT}/valid.csv")["id_code"]], "APTOS val")

VAL_DS, VAL_LOADER = DualViewDataset(VAL_CSV), None
VAL_LOADER = loader(VAL_DS, 16, False)
print(f"validation loader ready: {len(VAL_DS)} eyes")

## 7 — Models

**CORAL** gives monotone cumulative outputs `P(y>k)` by construction (ordered biases built from
non-negative softplus steps). Thresholds are initialised from the empirical marginals
`b_k = logit(P(Y>k))` — arbitrary initialisation collapses intermediate grades, because CORAL
compares one scalar per sample against all thresholds.

**InteractionFusion** takes `[z_m, z_d, |z_m − z_d|, z_m ⊙ z_d]` through a small MLP: a bare linear
fusion can only form a weighted sum and cannot represent cross-view interaction.

In [ ]:
import torchvision.models as tv

def pos_weights(csv, col="grade"):
    g = pd.read_csv(csv)[col].values
    w = []
    for k in range(NUM_THRESHOLDS):
        pos, neg = int((g > k).sum()), int((g <= k).sum())
        if pos == 0 or neg == 0: raise ValueError(f"degenerate threshold k={k}")
        w.append(math.sqrt(neg / pos) if POS_WEIGHT_MODE == "sqrt" else (neg / pos))
    return torch.tensor(w, dtype=torch.float32)

def init_thresholds(csv, col="grade", eps=1e-3):
    g = pd.read_csv(csv)[col].values
    out = []
    for k in range(NUM_THRESHOLDS):
        p = min(max(float((g > k).mean()), eps), 1 - eps)
        out.append(math.log(p / (1 - p)))
    return out

class CORALHead(nn.Module):
    def __init__(self, in_dim, num_classes=NUM_CLASSES, init=None):
        super().__init__()
        self.k = num_classes - 1
        self.fc = nn.Linear(in_dim, 1, bias=False)
        t = torch.tensor(list(init if init is not None else [-0.7 * i for i in range(self.k)]),
                         dtype=torch.float32)
        gaps = (t[:-1] - t[1:]).clamp_min(1e-4)
        self.base_bias = nn.Parameter(t[0].clone())
        self.bias_steps = nn.Parameter(torch.log(torch.expm1(gaps)).clone())
    def biases(self):
        steps = F.softplus(self.bias_steps)
        return self.base_bias - torch.cat([torch.zeros(1, device=steps.device), torch.cumsum(steps, 0)])
    def forward(self, z):
        logits = self.fc(z) + self.biases().unsqueeze(0)
        return logits, torch.sigmoid(logits)

class InteractionFusion(nn.Module):
    def __init__(self, feat_dim, hidden=None):
        super().__init__()
        hidden = hidden or feat_dim
        self.norm_in  = nn.LayerNorm(feat_dim * 4)
        self.proj     = nn.Linear(feat_dim * 4, hidden)
        self.act      = nn.ReLU(inplace=True)
        self.norm_out = nn.LayerNorm(hidden)
        self.out_dim  = hidden
    def forward(self, zm, zd):
        return self.norm_out(self.act(self.proj(
            self.norm_in(torch.cat([zm, zd, torch.abs(zm - zd), zm * zd], dim=1)))))

class DWSepBlock(nn.Module):
    """ReLU (not ReLU6): eager fuse_modules has no fuser for Conv-BN-ReLU6."""
    def __init__(self, i, o, stride=1):
        super().__init__()
        self.dw, self.bn1, self.act1 = nn.Conv2d(i, i, 3, stride, 1, groups=i, bias=False), nn.BatchNorm2d(i), nn.ReLU(inplace=True)
        self.pw, self.bn2, self.act2 = nn.Conv2d(i, o, 1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True)
    def forward(self, x): return self.act2(self.bn2(self.pw(self.act1(self.bn1(self.dw(x))))))
    def fuse(self, qat=False):
        fn = torch.ao.quantization.fuse_modules_qat if qat else torch.ao.quantization.fuse_modules
        fn(self, [["dw", "bn1", "act1"], ["pw", "bn2", "act2"]], inplace=True)

class LightBackbone(nn.Module):
    def __init__(self, channels=STUDENT_CHANNELS):
        super().__init__()
        ch = tuple(channels)
        self.stem_conv = nn.Conv2d(3, ch[0], 3, 2, 1, bias=False)
        self.stem_bn, self.stem_act = nn.BatchNorm2d(ch[0]), nn.ReLU(inplace=True)
        strides = [2 if i % 2 == 0 else 1 for i in range(len(ch) - 1)]
        self.blocks = nn.ModuleList([DWSepBlock(ch[i], ch[i + 1], strides[i]) for i in range(len(ch) - 1)])
        self.gap, self.out_dim = nn.AdaptiveAvgPool2d(1), ch[-1]
    def forward(self, x):
        x = self.stem_act(self.stem_bn(self.stem_conv(x)))
        for b in self.blocks: x = b(x)
        return self.gap(x).flatten(1)
    def fuse_model(self, qat=False):
        fn = torch.ao.quantization.fuse_modules_qat if qat else torch.ao.quantization.fuse_modules
        fn(self, [["stem_conv", "stem_bn", "stem_act"]], inplace=True)
        for b in self.blocks: b.fuse(qat=qat)

class DualViewBase(nn.Module):
    def forward(self, macula, disc):
        zm, zd = self.backbone(macula), self.backbone(disc)
        zf = self.fusion(zm, zd)
        ld, pd_ = self.main_head(zf)
        lm, pm = self.macula_head(zm)
        ldd, pdd = self.disc_head(zd)
        return {"p_dual": pd_, "logit_dual": ld, "p_macula": pm, "logit_macula": lm,
                "p_disc": pdd, "logit_disc": ldd, "z_fused": zf}
    def forward_single(self, x, which="macula"):
        z = self.backbone(x)
        logit, p = (self.macula_head if which == "macula" else self.disc_head)(z)
        return {"logit": logit, "p": p}
    def counterfactual_forward(self, macula, disc):
        """Same-head counterfactual: dual / macula-only / disc-only all pass through main_head, so
        their difference cannot be head discrepancy."""
        zm, zd = self.backbone(macula), self.backbone(disc)
        zero = torch.zeros_like(zm)
        _, p_dual = self.main_head(self.fusion(zm, zd))
        _, p_m = self.main_head(self.fusion(zm, zero))
        _, p_d = self.main_head(self.fusion(zero, zd))
        return {"p_dual": p_dual, "p_macula_cf": p_m, "p_disc_cf": p_d}

class Teacher(DualViewBase):
    def __init__(self, feat_dim=2048, init=None):
        super().__init__()
        bb = tv.resnet50(weights=tv.ResNet50_Weights.IMAGENET1K_V2); bb.fc = nn.Identity()
        self.backbone = bb
        self.fusion = InteractionFusion(feat_dim)
        self.main_head = CORALHead(self.fusion.out_dim, NUM_CLASSES, init)
        self.macula_head = CORALHead(feat_dim, NUM_CLASSES, init)
        self.disc_head = CORALHead(feat_dim, NUM_CLASSES, init)

class Student(DualViewBase):
    def __init__(self, backbone=None, init=None):
        super().__init__()
        self.backbone = backbone or LightBackbone()
        fd = self.backbone.out_dim
        self.fusion = InteractionFusion(fd)
        self.main_head = CORALHead(self.fusion.out_dim, NUM_CLASSES, init)
        self.macula_head = CORALHead(fd, NUM_CLASSES, init)
        self.disc_head = CORALHead(fd, NUM_CLASSES, init)
    def fuse_model(self, qat=False): self.backbone.fuse_model(qat=qat)

INIT_TH = init_thresholds(TRAIN_CSV)
APTOS_INIT_TH = init_thresholds(f"{APTOS_ROOT}/train_1.csv", col="diagnosis")
POS_W = pos_weights(TRAIN_CSV)
print("CORAL init thresholds:", [round(t, 3) for t in INIT_TH])
print("  implied P(y>k)     :", [round(float(torch.sigmoid(torch.tensor(t))), 3) for t in INIT_TH])
print("pos_weight (sqrt)    :", [round(float(w), 3) for w in POS_W])
print("APTOS thresholds     :", [round(t, 3) for t in APTOS_INIT_TH])

def param_count(m): return int(sum(p.numel() for p in m.parameters()))

In [ ]:
# ---- CORAL / fusion unit tests ----
def test_coral():
    h = CORALHead(16, NUM_CLASSES, INIT_TH)
    b = h.biases().detach()
    assert torch.all(b[:-1] >= b[1:]), "thresholds must be non-increasing"
    spread = float(b[0] - b[-1])
    assert spread > 1.5, f"threshold spread {spread:.2f} too small -- grades would collapse"
    _, p = h(torch.randn(32, 16))
    assert torch.all(p[:, :-1] >= p[:, 1:] - 1e-6), "P(y>k) must be non-increasing in k"
    emp = [float(torch.sigmoid(torch.tensor(t))) for t in INIT_TH]
    got = [float(torch.sigmoid(x)) for x in b]
    assert max(abs(a - c) for a, c in zip(emp, got)) < 1e-5, "init must match empirical marginals"
    f = InteractionFusion(8).eval()
    a, c = torch.randn(4, 8), torch.randn(4, 8)
    assert not torch.allclose(f(a, c), f(c, a)), "fusion must model interaction, not a sum"
    return spread

_spread = test_coral()
gate("Gate2a_CORAL", True, f"monotone, matches marginals, spread={_spread:.2f} logits; "
                           "fusion is view-order sensitive")

## 8 — Losses

`L = L_task + λ·L_aux + α·L_logitKD + β·L_CSD (+ γ·L_featKD)`

**CSD.** `Δ = p_dual − (p_macula + p_disc)/2` for teacher and student; both are divided by a FIXED
GLOBAL scale `s = E_train[|Δ_T|]`, estimated once from the frozen teacher.

A per-batch divisor would amplify batches whose teacher shift happens to be small and shrink those
whose shift is large — that silently re-weights samples relative to each other. One fixed scalar
fixes the gradient magnitude without touching the relative structure.

`Δ` is measured through three heads, so besides genuine complementarity it can carry head and
calibration discrepancy. It is an **operational proxy of the dual-view ordinal decision shift** —
never "pure anatomical complementarity". The same-head counterfactual ablation bounds that concern.

**Feature-KD** projects STUDENT → TEACHER against a detached target, so the regression target is
fixed. Projecting the other way lets the target drift as the projector learns, which would make the
main control for CSD's novelty weaker than it looks.

In [ ]:
def coral_loss(logits, labels, pos_weight=None):
    levels = torch.arange(NUM_THRESHOLDS, device=logits.device).unsqueeze(0)
    y_k = (labels.unsqueeze(1) > levels).float()
    return F.binary_cross_entropy_with_logits(logits, y_k, pos_weight=pos_weight)

def aux_loss(out, labels, pos_weight=None):
    return (coral_loss(out["logit_macula"], labels, pos_weight) +
            coral_loss(out["logit_disc"], labels, pos_weight))

def logit_kd_loss(logit_t, logit_s, tau=2.0):
    """No tau^2 factor, so alpha and tau are coupled -- do not claim independent temperature tuning."""
    return F.binary_cross_entropy(torch.sigmoid(logit_s / tau), torch.sigmoid(logit_t.detach() / tau))

def delta(p_dual, p_m, p_d): return p_dual - (p_m + p_d) / 2

CSD_SCALE, CSD_SCALE_CF = 1.0, 1.0        # set from the frozen teacher in Section 11

def csd_loss(t_out, s_out, variant="smoothl1_norm", scale=None, cf=False, tau=0.5, huber=1.0):
    km, kd = ("p_macula_cf", "p_disc_cf") if cf else ("p_macula", "p_disc")
    dt = delta(t_out["p_dual"].detach(), t_out[km].detach(), t_out[kd].detach())
    ds = delta(s_out["p_dual"], s_out[km], s_out[kd])
    s = max(float(scale if scale is not None else (CSD_SCALE_CF if cf else CSD_SCALE)), 1e-3)
    if variant == "smoothl1_norm":
        return F.smooth_l1_loss(ds / s, dt / s, beta=huber)
    if variant == "smoothl1":                       # unscaled -- ablation only
        return F.smooth_l1_loss(ds, dt, beta=huber)
    if variant == "magnitude_weighted_direction":
        mag = dt.norm(dim=1)
        w = (mag / mag.median().clamp_min(1e-6)).clamp(max=1.0)
        l_dir = ((1 - F.cosine_similarity(ds, dt, dim=1, eps=1e-6)) * w).sum() / w.sum().clamp_min(1e-6)
        return 0.5 * l_dir + 0.5 * F.smooth_l1_loss(ds / s, dt / s, beta=huber)
    if variant == "kl_softmax":                     # negative control: destroys magnitude info
        return F.kl_div(F.log_softmax(ds / tau, 1), F.softmax(dt / tau, 1), reduction="batchmean")
    raise ValueError(variant)

def feature_kd_loss(z_t, z_s, projector):
    return F.mse_loss(projector(z_s), z_t.detach())

def student_output(model, m, d, view):
    if view == "dual":        return model(m, d)
    if view == "macula_only": return model.forward_single(m, "macula")
    return model.forward_single(d, "disc")

def combined_loss(t_out, s_out, labels, view, alpha=0.0, beta=0.0, lambda_aux=0.5, tau_kd=2.0,
                  csd_variant="smoothl1_norm", pos_weight=None, gamma_feat=0.0, projector=None,
                  cf=False, t_cf=None, s_cf=None):
    task_logit = s_out["logit_dual"] if view == "dual" else s_out["logit"]
    l_task = coral_loss(task_logit, labels, pos_weight)
    total, log, comps = l_task, {"L_task": l_task.detach().item()}, {"task": l_task}
    if view == "dual":
        l_aux = aux_loss(s_out, labels, pos_weight)
        total = total + lambda_aux * l_aux
        log["L_aux"] = l_aux.detach().item(); comps["aux"] = lambda_aux * l_aux
        if alpha > 0:
            l_kd = logit_kd_loss(t_out["logit_dual"], s_out["logit_dual"], tau_kd)
            total = total + alpha * l_kd
            log["L_logit_KD"] = l_kd.detach().item(); comps["logit_kd"] = alpha * l_kd
        if beta > 0:
            l_csd = (csd_loss(t_cf, s_cf, csd_variant, cf=True) if cf
                     else csd_loss(t_out, s_out, csd_variant))
            total = total + beta * l_csd
            log["L_CSD"] = l_csd.detach().item(); comps["csd"] = beta * l_csd
        if gamma_feat > 0 and projector is not None:
            l_f = feature_kd_loss(t_out["z_fused"], s_out["z_fused"], projector)
            total = total + gamma_feat * l_f
            log["L_feat_KD"] = l_f.detach().item(); comps["feat_kd"] = gamma_feat * l_f
    log["L_total"] = total.detach().item()
    return total, log, comps

def grad_norms(comps, model):
    """Per-term gradient norms on the SHARED BACKBONE -- the only parameters every loss term reaches.
    Probing fusion+main_head instead reports 0 for the auxiliary term by construction."""
    params = [p for p in model.backbone.parameters() if p.requires_grad]
    out = {}
    for name, t in comps.items():
        if t is None or not getattr(t, "requires_grad", False): continue
        g = torch.autograd.grad(t, params, retain_graph=True, allow_unused=True)
        out[f"gnorm_{name}"] = sum(float(x.pow(2).sum()) for x in g if x is not None) ** 0.5
    if out.get("gnorm_task", 0) > 0:
        for k in ("csd", "aux", "logit_kd", "feat_kd"):
            if f"gnorm_{k}" in out: out[f"ratio_{k}_over_task"] = out[f"gnorm_{k}"] / out["gnorm_task"]
    return out

print("Losses defined.")

## 9 — Metrics

QWK is the primary metric, so it is checked against `sklearn.cohen_kappa_score(weights="quadratic",
labels=0..4)`. `labels=` matters: without it sklearn infers the label set from the data, so a model
that never predicts grade 4 would be scored on a 4×4 matrix while ours always uses 5×5.

In [ ]:
from sklearn.metrics import (confusion_matrix, f1_score, precision_recall_fscore_support,
                             balanced_accuracy_score, cohen_kappa_score)
from scipy import stats as sps

def fast_qwk(y_true, y_pred, K=NUM_CLASSES):
    y_true, y_pred = np.asarray(y_true, int), np.asarray(y_pred, int)
    O = np.zeros((K, K)); np.add.at(O, (y_true, y_pred), 1)
    w = (np.arange(K)[:, None] - np.arange(K)[None, :]) ** 2 / (K - 1) ** 2
    E = np.outer(np.bincount(y_true, minlength=K), np.bincount(y_pred, minlength=K)) / max(len(y_true), 1)
    den = (w * E).sum()
    return 1.0 - (w * O).sum() / den if den > 0 else 0.0

def test_qwk_matches_sklearn(n=100, seed=0):
    rng = np.random.default_rng(seed); labels = list(range(NUM_CLASSES)); worst = 0.0
    def cmp(yt, yp):
        nonlocal worst
        b = cohen_kappa_score(yt, yp, weights="quadratic", labels=labels)
        if np.isfinite(b): worst = max(worst, abs(fast_qwk(yt, yp) - b))
    for _ in range(n):
        m = int(rng.integers(30, 400)); yt = rng.integers(0, NUM_CLASSES, m)
        cmp(yt, np.clip(yt + rng.integers(-2, 3, m), 0, NUM_CLASSES - 1))
    y = rng.integers(0, NUM_CLASSES, 200)
    cmp(y, y.copy()); cmp(y, (NUM_CLASSES - 1) - y); cmp(y, np.zeros_like(y))
    cmp(y, np.clip(y, 0, NUM_CLASSES - 2)); cmp(np.array([0, 0, 4, 4]), np.array([0, 4, 0, 4]))
    return worst

_qwk_dev = test_qwk_matches_sklearn()
gate("Gate2b_QWK_Reference", _qwk_dev < 1e-10,
     f"max |fast_qwk - sklearn| = {_qwk_dev:.2e} over 105 cases "
     "(random, perfect, reversed, single-class, missing-grade, tiny)", blocking=BLOCK)

def calibration(p_cum, y_true, n_bins=10):
    """Pooled over the K-1 cumulative thresholds -- NOT conventional multiclass ECE, hence the name.
    Weighted-BCE training distorts these sigmoids, so they are ordinal threshold SCORES, not
    calibrated probabilities."""
    p = p_cum.detach().cpu().float()
    y = torch.as_tensor(np.asarray(y_true, int)).long()
    t = (y.unsqueeze(1) > torch.arange(p.shape[1]).unsqueeze(0)).float()
    pf, tf = p.flatten(), t.flatten()
    brier = float(((pf - tf) ** 2).mean())
    edges = torch.linspace(0, 1, n_bins + 1); ece = 0.0
    for i in range(n_bins):
        msk = (pf > edges[i]) & (pf <= edges[i + 1]) if i else (pf >= edges[i]) & (pf <= edges[i + 1])
        if msk.sum(): ece += float(msk.sum()) / pf.numel() * abs(float(pf[msk].mean()) - float(tf[msk].mean()))
    out = {"OrdinalThreshold_Brier": brier, "OrdinalThreshold_ECE": ece}
    for k in range(p.shape[1]):
        pk, tk = p[:, k], t[:, k]
        out[f"Threshold{k}_MAEProb"] = float((pk - tk).abs().mean())
    return out

def ordinal_violation_rate(p): return float((p[:, 1:] - p[:, :-1] > 0).float().mean())

def all_metrics(y_true, y_pred, p_cum=None, K=NUM_CLASSES):
    y_true, y_pred = np.asarray(y_true, int), np.asarray(y_pred, int)
    labels = list(range(K))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    pr, rc, f1, sup = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    m = {"QWK": fast_qwk(y_true, y_pred),
         "Accuracy": float((y_true == y_pred).mean()),
         "BalancedAccuracy": float(balanced_accuracy_score(y_true, y_pred)),
         "MacroPrecision": float(pr.mean()), "MacroRecall": float(rc.mean()),
         "MacroF1": float(f1_score(y_true, y_pred, average="macro", labels=labels, zero_division=0)),
         "WeightedF1": float(f1_score(y_true, y_pred, average="weighted", labels=labels, zero_division=0)),
         "MAE": float(np.mean(np.abs(y_true - y_pred))),
         "SevereErrorRate": float(np.mean(np.abs(y_true - y_pred) >= 2))}
    for g in labels:
        tp = cm[g, g]; fp = cm[:, g].sum() - tp; fn = cm[g, :].sum() - tp
        tn = cm.sum() - tp - fp - fn
        m[f"Precision_Grade{g}"] = float(pr[g]); m[f"Recall_Grade{g}"] = float(rc[g])
        m[f"F1_Grade{g}"] = float(f1[g]); m[f"Support_Grade{g}"] = int(sup[g])
        m[f"Specificity_Grade{g}"] = float(tn / (tn + fp)) if (tn + fp) else float("nan")
    if p_cum is not None:
        m["OrdinalViolationRate"] = ordinal_violation_rate(p_cum)
        m.update(calibration(p_cum, y_true))
    return m

def collapse_warnings(y_true, y_pred):
    w = []
    if len(np.unique(y_pred)) <= 2: w.append(f"only {len(np.unique(y_pred))} distinct grades predicted")
    for g in range(NUM_CLASSES):
        if (np.asarray(y_true) == g).sum() and not (np.asarray(y_pred) == g).sum():
            w.append(f"grade {g} NEVER predicted")
    return w

print("Metrics defined.")

In [ ]:
# ---- evaluation helpers ----
@torch.no_grad()
def predict(model, ld, device=None, view="dual", clusters=False):
    device = device or DEVICE
    model.eval(); yt, yp, ps, cid = [], [], [], []
    for b in ld:
        m, d = b["macula"].to(device), b["disc"].to(device)
        o = student_output(model, m, d, view)
        p = o["p_dual" if view == "dual" else "p"]
        yp.extend((p > 0.5).sum(1).cpu().tolist()); yt.extend(b["label"].tolist()); ps.append(p.cpu())
        if clusters: cid.extend(b["cluster_id"].tolist())
    out = (np.array(yt), np.array(yp), torch.cat(ps, 0))
    return out + (np.array(cid),) if clusters else out

@torch.no_grad()
def predict_cpu(model, ld):
    """One CPU pass returns dual AND both auxiliary heads. Quantized models are CPU-only, and a
    single pass avoids relying on forward_single."""
    yt, cid, pd_, pm_, pdd_ = [], [], [], [], []
    for b in ld:
        o = model(b["macula"].cpu(), b["disc"].cpu())
        pd_.append(o["p_dual"]); pm_.append(o["p_macula"]); pdd_.append(o["p_disc"])
        yt.extend(b["label"].tolist()); cid.extend(b["cluster_id"].tolist())
    P, M_, D_ = torch.cat(pd_), torch.cat(pm_), torch.cat(pdd_)
    return {"y_true": np.array(yt), "cluster_ids": np.array(cid), "p_dual": P,
            "y_pred": (P > 0.5).sum(1).numpy(), "y_pred_macula": (M_ > 0.5).sum(1).numpy(),
            "y_pred_disc": (D_ > 0.5).sum(1).numpy()}

def quick_qwk(model, ld, view="dual"):
    y, yp, _ = predict(model, ld, view=view); return fast_qwk(y, yp)

@torch.no_grad()
def dual_view_gain(model, ld):
    """G_aux: dual head vs this model's OWN auxiliary heads (shared backbone).

    ONE forward pass, not three: `forward()` already returns the dual head and both auxiliary heads,
    so the previous version re-ran the backbone twice for outputs it had already computed."""
    model.eval(); yt, pdl, pm, pdd = [], [], [], []
    for b in ld:
        o = model(b["macula"].to(DEVICE), b["disc"].to(DEVICE))
        pdl.append(o["p_dual"].cpu()); pm.append(o["p_macula"].cpu()); pdd.append(o["p_disc"].cpu())
        yt.extend(b["label"].tolist())
    y = np.array(yt)
    g = lambda t: (torch.cat(t) > 0.5).sum(1).numpy()
    qd, qm_, qdd_ = fast_qwk(y, g(pdl)), fast_qwk(y, g(pm)), fast_qwk(y, g(pdd))
    return {"QWK_dual": qd, "QWK_aux_macula": qm_, "QWK_aux_disc": qdd_,
            "DualViewGain_G_aux": qd - max(qm_, qdd_)}

def _nll(p_cum, y):
    levels = torch.arange(NUM_THRESHOLDS, device=p_cum.device).unsqueeze(0)
    y_k = (y.unsqueeze(1) > levels).float()
    p = p_cum.clamp(1e-6, 1 - 1e-6)
    return -(y_k * torch.log(p) + (1 - y_k) * torch.log(1 - p)).sum(1)

@torch.no_grad()
def shift_fidelity(teacher, student, ld, cf=False):
    """Did the SHIFT PATTERN transfer, independently of whether QWK moved?

    BenefitCorr is the strongest of the three: it correlates teacher and student per-sample fusion
    benefit B = NLL(p_agg) - NLL(p_dual). If complementarity really transferred, the student should
    benefit from dual-view on the SAME samples the teacher does."""
    teacher.eval(); student.eval()
    km, kd = ("p_macula_cf", "p_disc_cf") if cf else ("p_macula", "p_disc")
    smae, cos, bt, bs = [], [], [], []
    for b in ld:
        m, d, y = b["macula"].to(DEVICE), b["disc"].to(DEVICE), b["label"].to(DEVICE)
        to = teacher.counterfactual_forward(m, d) if cf else teacher(m, d)
        so = student.counterfactual_forward(m, d) if cf else student(m, d)
        dt, ds = delta(to["p_dual"], to[km], to[kd]), delta(so["p_dual"], so[km], so[kd])
        smae.append((ds - dt).abs().sum(1).cpu())
        cos.append(F.cosine_similarity(ds, dt, dim=1, eps=1e-6).cpu())
        for out, sink in ((to, bt), (so, bs)):
            sink.append((_nll((out[km] + out[kd]) / 2, y) - _nll(out["p_dual"], y)).cpu())
    smae, cos = torch.cat(smae), torch.cat(cos)
    a, b_ = torch.cat(bt).numpy(), torch.cat(bs).numpy()
    ok = a.std() > 1e-8 and b_.std() > 1e-8
    pear = float(sps.pearsonr(a, b_)[0]) if ok else float("nan")
    spear = float(sps.spearmanr(a, b_).statistic) if ok else float("nan")
    pfx = "CF_" if cf else ""
    return {f"{pfx}ShiftL1": float(smae.mean()),                    # mean L1 NORM of the difference
            f"{pfx}ShiftMAE": float(smae.mean() / NUM_THRESHOLDS),  # per-threshold mean abs error
            f"{pfx}CosAgree": float(cos.mean()), f"{pfx}BenefitCorr": pear,
            f"{pfx}BenefitCorrSpearman": spear}

def benchmark_latency(model, m, d, bench=None, on_cpu=True):
    """Median-of-medians over independent blocks; a shared Colab CPU makes a single block noisy."""
    bench = bench or BENCH
    torch.set_num_threads(bench["threads"])
    mm = (copy.deepcopy(model).to("cpu") if on_cpu else model).eval()
    a, b = m[:bench["batch_size"]].cpu(), d[:bench["batch_size"]].cpu()
    meds, allt = [], []
    with torch.no_grad():
        for _ in range(bench["warmup"]): mm(a, b)
        for _ in range(bench["repeats"]):
            ts = []
            for _ in range(bench["runs"]):
                t0 = time.perf_counter(); mm(a, b); ts.append((time.perf_counter() - t0) * 1000)
            allt += ts; meds.append(float(np.median(ts)))
    ts, meds = np.array(allt), np.array(meds)
    med = float(np.median(meds))
    return {"Latency_median_ms": med, "Latency_mean_ms": float(ts.mean()),
            "Latency_sd_ms": float(ts.std()), "Latency_p95_ms": float(np.percentile(ts, 95)),
            "Latency_p99_ms": float(np.percentile(ts, 99)),
            "Latency_block_IQR_ms": float(np.percentile(meds, 75) - np.percentile(meds, 25)),
            # one inference = one EYE = one (macula, disc) PAIR = two fundus images
            "Throughput_pairs_per_s": 1000.0 / med if med > 0 else float("nan"),
            "Throughput_images_per_s": 2000.0 / med if med > 0 else float("nan"),
            "bench_runs": bench["runs"], "bench_repeats": bench["repeats"]}

def state_dict_size_mb(model, tag):
    """Model size is ALWAYS a freshly serialized state_dict, FP32 and INT8 alike -- comparing a
    training checkpoint against a state_dict would measure two different kinds of file."""
    p = f"{CKPT_DIR}/size_probe/{tag}.pt"
    os.makedirs(os.path.dirname(p), exist_ok=True)
    torch_save({k: v.detach().cpu() for k, v in model.state_dict().items()}, p)
    return p, file_size_mb(p)

def measure_memory(build, m, d, n=20):
    try: import psutil
    except Exception: return {}
    import threading
    proc = psutil.Process(os.getpid())
    base = proc.memory_info().rss / 1024 ** 2
    mm = build(); loaded = proc.memory_info().rss / 1024 ** 2
    peak, stop = [loaded], threading.Event()
    def poll():
        while not stop.is_set():
            peak[0] = max(peak[0], proc.memory_info().rss / 1024 ** 2); time.sleep(0.005)
    th = threading.Thread(target=poll, daemon=True); th.start()
    try:
        with torch.no_grad():
            for _ in range(n): mm(m[:1].cpu(), d[:1].cpu())
    finally:
        stop.set(); th.join(timeout=1)
    after = proc.memory_info().rss / 1024 ** 2
    del mm
    return {"ModelLoadRSSDelta_MB": max(loaded - base, 0.0),
            "InferenceRSSDelta_MB": max(after - loaded, 0.0),
            "PeakRSS_during_inference_MB": max(peak[0], after),
            "RSS_scope": "process-level, includes the notebook"}

print("Evaluation helpers defined.")

## 10 — Smoke test (Gate 2c)

In [ ]:
def smoke_test():
    ds = DualViewDataset(TRAIN_CSV)
    b = next(iter(loader(ds, 8, True, seed=PRIMARY_SEED)))
    m, d, y = b["macula"].to(DEVICE), b["disc"].to(DEVICE), b["label"].to(DEVICE)
    teacher, student = Teacher(init=INIT_TH).to(DEVICE), Student(init=INIT_TH).to(DEVICE)
    ns, nt = param_count(student), param_count(teacher)
    print(f"Student {ns:,} params (backbone {param_count(student.backbone):,}) | "
          f"Teacher {nt:,} | compression {nt/ns:.0f}x")
    assert ns > 150_000, f"student too small ({ns:,}) -- would be capacity-capped"

    t_out = teacher(m, d)
    assert ordinal_violation_rate(t_out["p_dual"]) == 0.0, "CORAL monotonicity broken"
    for view in ("dual", "macula_only", "disc_only"):
        s_out = student_output(student, m, d, view)
        loss, log, comps = combined_loss(t_out, s_out, y, view, alpha=0.5, beta=1.0,
                                         pos_weight=POS_W.to(DEVICE))
        loss.backward(); student.zero_grad()
        print(f"  [{view}] loss={loss.item():.4f}")
    s_cf, t_cf = student.counterfactual_forward(m, d), teacher.counterfactual_forward(m, d)
    l, _, _ = combined_loss(t_out, student(m, d), y, "dual", alpha=0.5, beta=1.0, cf=True,
                            t_cf=t_cf, s_cf=s_cf, pos_weight=POS_W.to(DEVICE))
    l.backward(); student.zero_grad()

    _, _, comps = combined_loss(t_out, student(m, d), y, "dual", alpha=0.5, beta=1.0,
                                pos_weight=POS_W.to(DEVICE))
    gn = grad_norms(comps, student); student.zero_grad()
    ratio = gn.get("ratio_csd_over_task", 0.0)
    print("  gradient norms:", {k: round(v, 4) for k, v in gn.items()})
    assert ratio > 0.01, f"CSD gradient ratio {ratio:.5f} is negligible"
    assert gn.get("gnorm_aux", 0) > 0, "auxiliary loss must reach the shared backbone"

    student.eval(); student.fuse_model()
    del teacher, student
    return ns, nt, ratio

_ns, _nt, _ratio = smoke_test()
gate("Gate2c_Smoke", True, f"student={_ns:,} teacher={_nt:,}; forward/backward/fuse OK; "
                           f"CSD/task gradient ratio at beta=1 is {_ratio:.3f}")

## 11 — APTOS pretraining & teacher

Backbones are pretrained on APTOS with **their own** CORAL thresholds (APTOS has a different grade
marginal than DRTiD, so reusing DRTiD's would start the head in the wrong place).

The teacher warms up its heads with backbone **weights** frozen — note that BatchNorm running
statistics still adapt, which is deliberate domain adaptation and is stated as such rather than
called a "frozen backbone".

Every checkpoint is written **once, when the job finishes**, so a file that exists is a job that
completed. That is the whole resume story.

In [ ]:
def build_backbone(kind):
    if kind == "resnet50":
        bb = tv.resnet50(weights=tv.ResNet50_Weights.IMAGENET1K_V2); bb.fc = nn.Identity()
        return bb, 2048
    bb = LightBackbone()
    return bb, bb.out_dim

def pretrain_backbone(kind, cfg):
    out = f"{CKPT_DIR}/backbone_{kind}.pt"
    if os.path.exists(out):
        print(f"{kind}: checkpoint exists, skipping."); return out
    set_seed(PRIMARY_SEED)                       # BEFORE construction, so init is controlled
    backbone, feat = build_backbone(kind)
    head = CORALHead(feat, NUM_CLASSES, APTOS_INIT_TH)
    backbone, head = backbone.to(DEVICE), head.to(DEVICE)
    pw = pos_weights(f"{APTOS_ROOT}/train_1.csv", col="diagnosis").to(DEVICE)
    tl = loader(APTOSDataset(f"{APTOS_ROOT}/train_1.csv", f"{APTOS_ROOT}/train_images/train_images",
                             aptos_train_tf), cfg["batch_size"], True, PRIMARY_SEED)
    vl = loader(APTOSDataset(f"{APTOS_ROOT}/valid.csv", f"{APTOS_ROOT}/val_images/val_images",
                             aptos_eval_tf), cfg["batch_size"], False)
    opt = torch.optim.AdamW(list(backbone.parameters()) + list(head.parameters()),
                            lr=cfg["lr"], weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"], eta_min=cfg["lr"] * 0.02)
    stop = EarlyStopping(cfg["patience"])
    hist = []
    for ep in range(cfg["epochs"]):
        backbone.train(); head.train()
        for b in tqdm(tl, desc=f"[{kind}] ep{ep}", leave=False):
            loss = coral_loss(head(backbone(b["image"].to(DEVICE)))[0], b["label"].to(DEVICE), pw)
            opt.zero_grad(); loss.backward(); opt.step()
        sch.step()
        backbone.eval(); head.eval()
        pr, tg = [], []
        with torch.no_grad():
            for b in vl:
                _, p = head(backbone(b["image"].to(DEVICE)))
                pr.extend((p > 0.5).sum(1).cpu().tolist()); tg.extend(b["label"].tolist())
        q = fast_qwk(tg, pr); hist.append({"epoch": ep, "val_qwk": q})
        print(f"  ep{ep}: val_QWK={q:.4f}")
        if stop.step(q, backbone, ep): break
    pd.DataFrame(hist).to_csv(f"{LOG_DIR}/pretrain_{kind}.csv", index=False)
    assert stop.best > 0.0, f"{kind} pretraining QWK <= 0 -- worse than a majority baseline"
    torch_save({"model_state": stop.best_state, "kind": kind, "best_val_qwk": float(stop.best),
                "epochs_run": len(hist), "best_epoch": stop.best_epoch, **cfg}, out)
    print(f"{kind} done. best val QWK={stop.best:.4f} @ep{stop.best_epoch} "
          f"({len(hist)}/{cfg['epochs']} epochs run)")
    return out

R50_CKPT   = pretrain_backbone("resnet50", APTOS_R50_CFG)
LIGHT_CKPT = pretrain_backbone("lightweight", APTOS_LIGHT_CFG)

In [ ]:
def train_teacher(cfg):
    out = f"{CKPT_DIR}/teacher.pt"
    if os.path.exists(out):
        print("teacher: checkpoint exists, skipping."); return out
    set_seed(PRIMARY_SEED)
    model = Teacher(init=INIT_TH).to(DEVICE)
    model.backbone.load_state_dict(load_weights(R50_CKPT, DEVICE))
    pw = POS_W.to(DEVICE)
    tl = loader(DualViewDataset(TRAIN_CSV, paired=paired_tf), cfg["batch_size"], True, PRIMARY_SEED)
    vl = loader(DualViewDataset(VAL_CSV), cfg["batch_size"], False)
    stop = EarlyStopping(cfg["patience"])
    hist, state = [], {"ep": 0}

    def run(epochs, lr):
        opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                lr=lr, weight_decay=1e-4)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs, 1), eta_min=lr * 0.02)
        for _ in range(epochs):
            model.train()
            for b in tqdm(tl, desc=f"[teacher] ep{state['ep']}", leave=False):
                m, d, y = b["macula"].to(DEVICE), b["disc"].to(DEVICE), b["label"].to(DEVICE)
                o = model(m, d)
                loss = coral_loss(o["logit_dual"], y, pw) + cfg["lambda_aux"] * aux_loss(o, y, pw)
                opt.zero_grad(); loss.backward(); opt.step()
            sch.step()
            q = quick_qwk(model, vl)
            hist.append({"epoch": state["ep"], "val_qwk": q})
            print(f"  ep{state['ep']}: val_QWK={q:.4f}")
            stopped = stop.step(q, model, state["ep"]); state["ep"] += 1
            if stopped: break

    # stage 1 -- head warm-up: backbone WEIGHTS frozen; BN statistics still adapt (deliberate)
    for p in model.backbone.parameters(): p.requires_grad = False
    run(cfg["freeze_epochs"], cfg["freeze_lr"])
    stop.restore(model)                       # fine-tune from the best warm-up weights
    stop.bad = 0                              # the second stage gets its own patience budget
    for p in model.backbone.parameters(): p.requires_grad = True
    run(cfg["finetune_epochs"], cfg["finetune_lr"])

    pd.DataFrame(hist).to_csv(f"{LOG_DIR}/teacher.csv", index=False)
    torch_save({"model_state": stop.best_state, "best_val_qwk": float(stop.best),
                "epochs_run": len(hist), "best_epoch": stop.best_epoch, **cfg}, out)
    print(f"Teacher done. best val QWK={stop.best:.4f} @ep{stop.best_epoch} "
          f"({len(hist)} epochs run)")
    return out

TEACHER_CKPT = train_teacher(TEACHER_CFG)

_teacher = None
def get_teacher():
    global _teacher
    if _teacher is None:
        _teacher = Teacher(init=INIT_TH).to(DEVICE)
        _teacher.load_state_dict(load_weights(TEACHER_CKPT, DEVICE)); _teacher.eval()
        for p in _teacher.parameters(): p.requires_grad = False
    return _teacher

In [ ]:
# ---- Gate 3: the teacher must actually gain from two views ----
# CSD distils a dual-view shift. Without a teacher dual-view advantage there is no shift worth
# transferring, and every downstream CSD result becomes uninterpretable.
_g = dual_view_gain(get_teacher(), VAL_LOADER)
print("Teacher dual-view gain (validation):", {k: round(v, 4) for k, v in _g.items()})
gate("Gate3_TeacherDualView", _g["DualViewGain_G_aux"] > 0,
     f"QWK_dual={_g['QWK_dual']:.4f} vs max(aux)={max(_g['QWK_aux_macula'], _g['QWK_aux_disc']):.4f} "
     f"-> G_aux={_g['DualViewGain_G_aux']:+.4f}", blocking=BLOCK)

# ---- fixed global CSD scale, from the FROZEN teacher over TRAIN ----
@torch.no_grad()
def global_delta_scale(teacher, ld, cf=False):
    tot, n = 0.0, 0
    km, kd = ("p_macula_cf", "p_disc_cf") if cf else ("p_macula", "p_disc")
    for b in ld:
        m, d = b["macula"].to(DEVICE), b["disc"].to(DEVICE)
        o = teacher.counterfactual_forward(m, d) if cf else teacher(m, d)
        dt = delta(o["p_dual"], o[km], o[kd])
        tot += float(dt.abs().sum()); n += dt.numel()
    return max(tot / max(n, 1), 1e-3)

_scale_ld = loader(DualViewDataset(TRAIN_CSV), 16, False)
CSD_SCALE    = global_delta_scale(get_teacher(), _scale_ld, cf=False)
CSD_SCALE_CF = global_delta_scale(get_teacher(), _scale_ld, cf=True)
print(f"CSD global scale E_train[|delta_T|] = {CSD_SCALE:.6f} (counterfactual {CSD_SCALE_CF:.6f})")
save_json({"csd_scale": CSD_SCALE, "csd_scale_cf": CSD_SCALE_CF}, f"{CFG_DIR}/csd_scale.json")

# ---- Gate 4a: is there a shift to distil at all? ----
@torch.no_grad()
def delta_stats(teacher, ld):
    norms = []
    for b in ld:
        o = teacher(b["macula"].to(DEVICE), b["disc"].to(DEVICE))
        norms.append(delta(o["p_dual"], o["p_macula"], o["p_disc"]).abs().sum(1).cpu())
    n = torch.cat(norms)
    pd.DataFrame({"delta_L1": n.numpy()}).to_csv(f"{MET_DIR}/teacher_delta_distribution.csv", index=False)
    return {"mean_L1": float(n.mean()), "median_L1": float(n.median()),
            "frac_gt_0.02": float((n > 0.02).float().mean())}

DELTA_STATS = delta_stats(get_teacher(), VAL_LOADER)
# NUMERICAL SANITY ONLY. This says the shift is not numerically negligible; it is NOT evidence that
# complementarity exists. That case rests on the teacher's dual-view gain, the shift distribution,
# the mechanism-fidelity metrics and the downstream predictive result.
gate("Gate4a_CSD_Signal", DELTA_STATS["mean_L1"] >= 1e-3,
     ", ".join(f"{k}={v:.4f}" for k, v in DELTA_STATS.items()) + " | numerical sanity only",
     blocking=BLOCK)

## 12 — Student trainer

One function drives every student condition, so training logic cannot drift between them. Per-epoch
loss values **and per-component gradient norms** are logged: loss values alone cannot show that a
term influences learning — a term can be numerically present and contribute no gradient.

In [ ]:
def train_student(run_name, seed, view="dual", alpha=0.0, beta=0.0, tau_kd=2.0, gamma_feat=0.0,
                  csd_variant="smoothl1_norm", cf=False, cfg=None):
    cfg = cfg or STUDENT_CFG
    out = f"{CKPT_DIR}/student/{run_name}/seed{seed}.pt"
    if os.path.exists(out):
        print(f"[{run_name}|s{seed}] exists, skipping."); return out
    os.makedirs(os.path.dirname(out), exist_ok=True)
    conf = dict(run_name=run_name, seed=seed, view=view, alpha=alpha, beta=beta, tau_kd=tau_kd,
                gamma_feat=gamma_feat, csd_variant=csd_variant, counterfactual=cf, **cfg)

    set_seed(seed)                                  # BEFORE construction
    # The teacher is only needed when a distillation term actually consumes it. The single-view
    # baselines and dual_no_distill were paying a full dual ResNet-50 forward per batch for an
    # output that was then discarded -- 15 of the ~100 jobs in the run.
    needs_teacher = (alpha > 0) or (beta > 0) or (gamma_feat > 0)
    teacher = get_teacher() if needs_teacher else None
    student = Student(init=INIT_TH).to(DEVICE)
    student.backbone.load_state_dict(load_weights(LIGHT_CKPT, DEVICE))
    proj, params = None, list(student.parameters())
    if gamma_feat > 0:
        proj = nn.Linear(student.fusion.out_dim, teacher.fusion.out_dim).to(DEVICE)
        params += list(proj.parameters())
    if not needs_teacher:
        print(f"  [{run_name}|s{seed}] no distillation term -- teacher forward skipped")

    pw = POS_W.to(DEVICE)
    tl = loader(DualViewDataset(TRAIN_CSV, paired=paired_tf), cfg["batch_size"], True, seed)
    vl = loader(DualViewDataset(VAL_CSV), cfg["batch_size"], False)
    opt = torch.optim.AdamW(params, lr=cfg["lr"], weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"], eta_min=cfg["lr"] * 0.02)
    stop, hist = EarlyStopping(cfg["patience"]), []

    for ep in range(cfg["epochs"]):
        student.train(); logs, gn = [], {}
        for i, b in enumerate(tqdm(tl, desc=f"[{run_name}|s{seed}] ep{ep}", leave=False)):
            m, d, y = b["macula"].to(DEVICE), b["disc"].to(DEVICE), b["label"].to(DEVICE)
            t_out, t_cf = None, None
            if needs_teacher:
                with torch.no_grad():
                    t_out = teacher(m, d)
                    t_cf = teacher.counterfactual_forward(m, d) if cf else None
            s_out = student_output(student, m, d, view)
            s_cf = student.counterfactual_forward(m, d) if (cf and view == "dual") else None
            loss, log, comps = combined_loss(t_out, s_out, y, view, alpha=alpha, beta=beta,
                                             lambda_aux=cfg["lambda_aux"], tau_kd=tau_kd,
                                             csd_variant=csd_variant, pos_weight=pw,
                                             gamma_feat=gamma_feat, projector=proj,
                                             cf=cf, t_cf=t_cf, s_cf=s_cf)
            if i == 0 and view == "dual":
                gn = grad_norms(comps, student); student.zero_grad(set_to_none=True)
            opt.zero_grad(); loss.backward(); opt.step()
            logs.append(log)
        sch.step()
        q = quick_qwk(student, vl, view)
        row = {k: float(np.mean([l[k] for l in logs if k in l])) for k in logs[0]}
        row.update({"epoch": ep, "val_qwk": q, "lr": sch.get_last_lr()[0], **gn})
        hist.append(row)
        print(f"  ep{ep}: val_QWK={q:.4f} " +
              str({k: round(v, 4) for k, v in row.items() if k.startswith("L_")}) +
              (f" grad={{k.replace('gnorm_',''): round(v,3) for k,v in gn.items()}}" if gn else ""))
        if stop.step(q, student, ep): break

    h = pd.DataFrame(hist); h.to_csv(f"{LOG_DIR}/{run_name}_seed{seed}.csv", index=False)
    assert stop.best_state is not None, f"[{run_name}|s{seed}] produced no checkpoint"
    torch_save({"model_state": stop.best_state, "val_qwk": float(stop.best), "seed": seed,
                "epochs_run": len(hist), "best_epoch": stop.best_epoch, "config": conf}, out)
    print(f"[{run_name}|s{seed}] best val QWK={stop.best:.4f} @ep{stop.best_epoch} "
          f"({len(hist)}/{cfg['epochs']} epochs)")
    del student, proj
    return out

def load_student(ckpt):
    m = Student(init=INIT_TH).to(DEVICE)
    m.load_state_dict(load_weights(ckpt, DEVICE)); m.eval()
    return m

print("Student trainer defined.")

## 13 — Baselines, grids, final conditions

**Grid protocol.** Each candidate is scored as the MEAN validation QWK over 3 tuning seeds, and a
candidate is valid only if **all 3 seeds completed** — a configuration averaged over 2 seeds is not
comparable with one averaged over 3.

**Additivity.** Feature-KD and CSD inherit `alpha`/`tau` from the logit-KD winner and tune only
their own coefficient, so the last rung of the ladder differs by exactly one term.

In [ ]:
# Single-view baselines on ALL core seeds, so the independent dual-view gain can be formed WITHIN a
# seed rather than against the mean of models trained on different seeds.
for s in SEEDS_CORE:
    train_student("macula_only", s, view="macula_only")
    train_student("disc_only",   s, view="disc_only")
for s in SEEDS_CORE:
    train_student("dual_no_distill", s, view="dual", alpha=0.0, beta=0.0)

In [ ]:
def run_grid(tag, grid, base, seeds=None):
    """Trains every candidate on every tuning seed; selects on the MEAN validation QWK."""
    seeds = seeds or SEEDS_TUNING
    rows, per_seed, invalid = [], [], []
    for combo in grid:
        name = tag + "_" + "_".join(f"{k}{v}" for k, v in sorted(combo.items()))
        qs = []
        for sd in seeds:
            try:
                ck = train_student(name, sd, view="dual", **{**base, **combo})
                mdl = load_student(ck)
                y, yp, pc = predict(mdl, VAL_LOADER)
                m = all_metrics(y, yp, pc); del mdl
            except Exception as e:
                print(f"  !! {name} seed {sd} FAILED: {e!r}"); continue
            qs.append(m); per_seed.append({**combo, "run_name": name, "seed": sd,
                                           "val_QWK": m["QWK"], "val_MacroF1": m["MacroF1"]})
        if len(qs) != len(seeds):
            invalid.append({"run_name": name, "n_ok": len(qs), "n_required": len(seeds)})
            print(f"  !! {name} INVALID: {len(qs)}/{len(seeds)} seeds"); continue
        rows.append({**combo, "run_name": name, "n_seeds": len(seeds),
                     "val_QWK_mean": float(np.mean([m["QWK"] for m in qs])),
                     "val_QWK_sd": float(np.std([m["QWK"] for m in qs], ddof=1)) if len(qs) > 1 else 0.0,
                     "val_MacroF1_mean": float(np.mean([m["MacroF1"] for m in qs])),
                     "val_SER_mean": float(np.mean([m["SevereErrorRate"] for m in qs])),
                     "val_MAE_mean": float(np.mean([m["MAE"] for m in qs]))})
    gate(f"Gate5_Grid_{tag}", len(rows) == len(grid),
         f"{len(rows)}/{len(grid)} candidates completed all {len(seeds)} tuning seeds"
         + (f"; invalid={[r['run_name'] for r in invalid]}" if invalid else ""), blocking=BLOCK)
    if not rows: raise GateFailure(f"{tag}: no candidate completed every tuning seed")
    df = (pd.DataFrame(rows)
          .sort_values(["val_QWK_mean", "val_MacroF1_mean", "val_SER_mean", "val_MAE_mean"],
                       ascending=[False, False, True, True]).reset_index(drop=True))
    df.to_csv(f"{TAB_DIR}/table_01_grid_{tag}.csv", index=False)
    pd.DataFrame(per_seed).to_csv(f"{TAB_DIR}/table_01_grid_{tag}_per_seed.csv", index=False)
    print(f"--- {tag}: mean validation QWK over seeds {seeds} ---")
    print(df.to_string(index=False))
    return df

GRID_KD = run_grid("logitkd", GRID_LOGITKD, {"beta": 0.0})
BEST_KD = {"alpha": float(GRID_KD.iloc[0]["alpha"]), "tau_kd": float(GRID_KD.iloc[0]["tau_kd"])}
print("selected logit-KD:", BEST_KD, f"(mean val QWK={GRID_KD.iloc[0]['val_QWK_mean']:.4f})")
for s in SEEDS_CORE:
    train_student("dual_logitkd", s, view="dual", beta=0.0, **BEST_KD)

In [ ]:
# Feature-KD: alpha/tau FROZEN at the logit-KD winner, only gamma is searched.
GRID_FK = run_grid("featkd", GRID_FEATKD, {"beta": 0.0, **BEST_KD})
BEST_FEAT = {"gamma_feat": float(GRID_FK.iloc[0]["gamma_feat"])}
print("selected feature-KD:", BEST_FEAT, f"(mean val QWK={GRID_FK.iloc[0]['val_QWK_mean']:.4f})")
for s in SEEDS_CORE:
    train_student("dual_featkd", s, view="dual", beta=0.0, **BEST_KD, **BEST_FEAT)

In [ ]:
# CSD: alpha/tau also FROZEN at the logit-KD winner; only beta and the variant are searched, so
# "logit-KD" and "logit-KD + CSD" differ by exactly one term.
GRID_CS = run_grid("csd", GRID_CSD, {**BEST_KD})
BEST_CSD_VARIANT = GRID_CS.iloc[0]["csd_variant"]
BEST_BETA = float(GRID_CS.iloc[0]["beta"])
BEST_ALPHA = float(BEST_KD["alpha"])
print(f"selected CSD: variant={BEST_CSD_VARIANT} beta={BEST_BETA} "
      f"(alpha={BEST_ALPHA}, tau={BEST_KD['tau_kd']} inherited) "
      f"| mean val QWK={GRID_CS.iloc[0]['val_QWK_mean']:.4f}")
save_json({"logitkd": BEST_KD, "featkd": BEST_FEAT, "csd_variant": BEST_CSD_VARIANT,
           "csd_beta": BEST_BETA, "alpha_source": "inherited from the logit-KD winner"},
          f"{CFG_DIR}/selected_hyperparameters.json")

for s in SEEDS_CORE:
    train_student("dual_csd", s, view="dual", beta=BEST_BETA, csd_variant=BEST_CSD_VARIANT, **BEST_KD)

In [ ]:
# ---- Gate 4b: does the SELECTED CSD configuration produce usable gradient? ----
# The smoke test ran before the teacher and the global scale existed. This re-measures with the
# frozen teacher, the real scale, and the configuration actually chosen.
def csd_gradient_check(betas, alpha, tau_kd):
    set_seed(PRIMARY_SEED)
    probe = Student(init=INIT_TH).to(DEVICE)
    probe.backbone.load_state_dict(load_weights(LIGHT_CKPT, DEVICE))
    b = next(iter(VAL_LOADER))
    m, d, y = b["macula"].to(DEVICE), b["disc"].to(DEVICE), b["label"].to(DEVICE)
    with torch.no_grad(): t_out = get_teacher()(m, d)
    ratios = {}
    for beta in betas:
        _, _, comps = combined_loss(t_out, probe(m, d), y, "dual", alpha=alpha, beta=beta,
                                    tau_kd=tau_kd, lambda_aux=STUDENT_CFG["lambda_aux"],
                                    pos_weight=POS_W.to(DEVICE))
        g = grad_norms(comps, probe); probe.zero_grad(set_to_none=True)
        ratios[beta] = g.get("ratio_csd_over_task", 0.0)
    del probe
    return ratios

_ratios = csd_gradient_check([BEST_BETA], BEST_ALPHA, BEST_KD["tau_kd"])
_r = _ratios[BEST_BETA]
save_json({"csd_scale": CSD_SCALE, "selected_beta": BEST_BETA, "alpha": BEST_ALPHA,
           "ratio_csd_over_task": _r}, f"{MET_DIR}/csd_gradient_diagnostic.json")
# [0.01, 10] is an engineering SANITY BAND, not a validated threshold: below it the CSD term is
# numerically irrelevant, above it CSD would swamp the task loss.
gate("Gate4b_SelectedCSD_Gradient", np.isfinite(_r) and 0.01 <= _r <= 10.0,
     f"selected beta={BEST_BETA}: CSD/task shared-backbone gradient ratio = {_r:.4f} "
     "(sanity band [0.01, 10])", blocking=BLOCK)

In [ ]:
# ---- CSD formulation ablations: SAME alpha/tau/beta, only the formulation changes ----
_ABL = dict(view="dual", alpha=BEST_ALPHA, tau_kd=BEST_KD["tau_kd"], beta=BEST_BETA)
for s in SEEDS_ABL:
    train_student("abl_csd_raw_smoothl1", s, csd_variant="smoothl1", **_ABL)
for s in SEEDS_ABL:
    train_student("abl_csd_kl_softmax", s, csd_variant="kl_softmax", **_ABL)
for s in SEEDS_ABL:
    train_student("abl_csd_counterfactual", s, csd_variant=BEST_CSD_VARIANT, cf=True, **_ABL)

## 14 — Model selection (validation only)

**Two-stage**, because picking the single best `(condition, seed)` row out of 20 runs is a maximum
over noisy estimates — a seed lottery whose winner also carries an upward-biased validation score.

```
STAGE 1 (statistical)  method M* = argmax_m mean_s QWK_val(m, s)     <- compares METHODS
STAGE 2 (operational)  checkpoint s* = argmax_s QWK_val(M*, s)       <- picks a deployable file
```

Ties within 0.005 resolve by Macro-F1 ↑ → severe-error ↓ → MAE ↓ → simpler method.
**The test set is not consulted at either stage.**

In [ ]:
def collect_val_scores():
    rows = []
    for cond in CORE_CONDITIONS:
        for s in SEEDS_CORE:
            ck = f"{CKPT_DIR}/student/{cond}/seed{s}.pt"
            if not os.path.exists(ck): continue
            mdl = load_student(ck)
            y, yp, pc = predict(mdl, VAL_LOADER)
            m = all_metrics(y, yp, pc); del mdl
            rows.append({"condition": cond, "seed": s, "checkpoint": ck,
                         "n_distinct_pred": int(len(np.unique(yp))),
                         **{k: m[k] for k in ("QWK", "MacroF1", "SevereErrorRate", "MAE",
                                              "Accuracy", "OrdinalViolationRate")}})
    return pd.DataFrame(rows)

VAL_SCORES = collect_val_scores()
VAL_SCORES.to_csv(f"{TAB_DIR}/table_02_validation_scores.csv", index=False)
print(VAL_SCORES.groupby("condition")[["QWK", "MacroF1", "SevereErrorRate", "MAE"]]
      .agg(["mean", "std"]).round(4).to_string())

# ---- completeness: every core method needs every core seed ----
_missing = {c: sorted(set(SEEDS_CORE) - set(VAL_SCORES[VAL_SCORES.condition == c].seed))
            for c in CORE_CONDITIONS}
_missing = {c: v for c, v in _missing.items() if v}
gate("Gate6a_CoreSeedCompleteness", not _missing,
     f"all core conditions have {len(SEEDS_CORE)}/{len(SEEDS_CORE)} seeds" if not _missing
     else f"MISSING {_missing}", blocking=BLOCK)

# ---- viability, on VALIDATION and BEFORE the test set is opened ----
_bad = [f"{r.condition}|s{r.seed}" for r in VAL_SCORES.itertuples()
        if not (np.isfinite(r.QWK) and np.isfinite(r.MacroF1)) or r.n_distinct_pred <= 2]
gate("Gate6b_ValidationViability", not _bad,
     f"{len(VAL_SCORES)} core runs finite and predicting >2 distinct grades" if not _bad
     else f"{len(_bad)} run(s) non-finite or collapsed: {_bad[:5]}", blocking=BLOCK)

# CORAL constructs monotone thresholds, so any violation is an integrity failure, not a metric.
_ovr = VAL_SCORES["OrdinalViolationRate"].max()
gate("Gate6c_OrdinalMonotonicity", float(_ovr) <= 1e-6,
     f"max ordinal violation rate = {_ovr:.2e}", blocking=BLOCK)

In [ ]:
def select_method(df):
    agg = (df.groupby("condition")
             .agg(QWK_mean=("QWK", "mean"), QWK_sd=("QWK", "std"), n=("QWK", "size"),
                  MacroF1=("MacroF1", "mean"), SER=("SevereErrorRate", "mean"), MAE=("MAE", "mean"))
             .reset_index())
    agg["simplicity"] = agg["condition"].map({c: i for i, c in enumerate(CORE_CONDITIONS)}).fillna(99)
    agg = agg.sort_values("QWK_mean", ascending=False).reset_index(drop=True)
    tied = agg[agg["QWK_mean"] >= agg.iloc[0]["QWK_mean"] - SELECTION_TIE_EPS]
    if len(tied) > 1:
        print(f"  {len(tied)} methods within {SELECTION_TIE_EPS} mean QWK -- tie-break chain applied")
        tied = tied.sort_values(["MacroF1", "SER", "MAE", "simplicity"],
                                ascending=[False, True, True, True])
    return tied.iloc[0], agg

def select_checkpoint(df, cond):
    d = df[df.condition == cond].sort_values("QWK", ascending=False).reset_index(drop=True)
    tied = d[d["QWK"] >= d.iloc[0]["QWK"] - SELECTION_TIE_EPS]
    if len(tied) > 1:
        tied = tied.sort_values(["MacroF1", "SevereErrorRate", "MAE"], ascending=[False, True, True])
    return tied.iloc[0]

METHOD_ROW, METHOD_TABLE = select_method(VAL_SCORES)
METHOD_TABLE.to_csv(f"{TAB_DIR}/table_02b_method_selection.csv", index=False)
print("Stage 1 -- method selection on MEAN validation QWK:")
print(METHOD_TABLE.round(4).to_string(index=False))

BEST_CONDITION = METHOD_ROW["condition"]
BEST_ROW = select_checkpoint(VAL_SCORES, BEST_CONDITION)
BEST_SEED, BEST_CKPT = int(BEST_ROW["seed"]), BEST_ROW["checkpoint"]
print(f"\nStage 1: method M* = {BEST_CONDITION} (mean val QWK={METHOD_ROW['QWK_mean']:.4f})")
print(f"Stage 2: deployment checkpoint = seed {BEST_SEED} (val QWK={BEST_ROW['QWK']:.4f})")

# Every core seed of the selected method is an RQ2 base model, so FP32_s -> PTQ_s / QAT_s /
# FP32FT_s / FT-PTQ_s is an exactly matched set per seed.
RQ2_BASE = {s: f"{CKPT_DIR}/student/{BEST_CONDITION}/seed{s}.pt" for s in SEEDS_CORE
            if os.path.exists(f"{CKPT_DIR}/student/{BEST_CONDITION}/seed{s}.pt")}
gate("Gate6d_RQ2BaseCompleteness", set(RQ2_BASE) == set(SEEDS_CORE),
     f"{len(RQ2_BASE)}/{len(SEEDS_CORE)} base checkpoints for {BEST_CONDITION}", blocking=BLOCK)

# The best CSD artifact is tracked separately: even if M* is not CSD, the paper still needs it.
_csd = VAL_SCORES[VAL_SCORES.condition == "dual_csd"]
BEST_CSD_SEED = int(select_checkpoint(VAL_SCORES, "dual_csd")["seed"]) if len(_csd) else PRIMARY_SEED
BEST_CSD_CKPT = f"{CKPT_DIR}/student/dual_csd/seed{BEST_CSD_SEED}.pt"
save_json({"method": BEST_CONDITION, "seed": BEST_SEED, "checkpoint": BEST_CKPT,
           "method_mean_val_qwk": float(METHOD_ROW["QWK_mean"]),
           "best_val_qwk": float(BEST_ROW["QWK"]), "best_csd_seed": BEST_CSD_SEED,
           "rule": "stage 1 mean validation QWK (method); stage 2 validation QWK (checkpoint)",
           "data": "DRTiD validation only -- the test set is not consulted"},
          f"{CFG_DIR}/model_selection.json")

## 15 — Quantization (RQ2)

**Matched scope.** RQ2 asks *PTQ vs QAT*, which is only answerable if both quantize the same
operators. Both use eager **backbone-only** quantization: same wrapper, same fused Conv-BN-ReLU set,
same qconfig family. `torch.cat`, LayerNorm and CORAL's cumsum/softplus have no eager INT8 kernels,
so the artifact is an **INT8 backbone with FP32 fusion and ordinal heads** (mixed precision).

**Matched per seed.** Every variant derives from the same `FP32_s`:

```
                  ┌── PTQ_s            (static calibration, no gradient)
FP32_s ───────────┼── QAT_s            (quantization-aware fine-tuning)
                  ├── FP32FT_s         (identical prepared graph, fake-quant OFF)
                  └── FP32FT-plain_s ─► FT-PTQ_s   (ordinary fine-tune, THEN calibration)
```

| Comparison | Question |
|---|---|
| PTQ vs FP32 | what does static INT8 cost? |
| QAT vs FP32 | what does quantization-aware training recover? |
| QAT vs FP32-FT | is the QAT gain more than extra fine-tuning? |
| QAT vs FT-PTQ | is adapting to quantization noise better than fine-tuning first and quantizing after? |

**Calibration is deterministic**: `shuffle=False` over the entire 800-eye training split, with a
hashed manifest, so re-running PTQ produces the same INT8 model.

**QAT is selected on the CONVERTED INT8 model** each epoch — that is the artifact that ships. Its
objective is task supervision only: no KD or CSD continues into quantization, so RQ2 measures
quantization rather than a second distillation experiment.

In [ ]:
from torch.ao.quantization import (prepare, convert, prepare_qat, get_default_qconfig,
                                   get_default_qat_qconfig, QuantStub, DeQuantStub,
                                   disable_fake_quant, disable_observer)

class QuantizableBackbone(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.quant, self.backbone, self.dequant = QuantStub(), backbone, DeQuantStub()
    def forward(self, x): return self.dequant(self.backbone(self.quant(x)))

def quantized_module_paths(model):
    """The sorted NAMES of quantized modules -- an exact operator set, not just a count.
    `PTQ ops == QAT ops == 15` does not prove the same 15 operators were quantized."""
    return sorted(n for n, m in model.named_modules() if "quantized" in type(m).__module__.lower())

def count_quantized(model):
    return sum(1 for m in model.modules() if "quantized" in type(m).__module__.lower())

def eligible_backbone_ops(model):
    """Denominator scoped to the DECLARED scope: counting every Conv/Linear while only the backbone
    is quantized would make a correct run look partial."""
    return sum(1 for n, m in model.named_modules()
               if isinstance(m, (nn.Conv2d, nn.Linear)) and n.startswith("backbone"))

def fresh_student(ckpt):
    m = Student(init=INIT_TH)
    m.load_state_dict(load_weights(ckpt, "cpu"))
    return m.to("cpu").eval()

# ---- deterministic calibration set ----
def calibration_manifest():
    df = pd.read_csv(TRAIN_CSV)[["record_id", "macula_filename", "disc_filename", "grade"]]
    p = f"{CFG_DIR}/ptq_calibration_manifest.csv"
    df.to_csv(p, index=False)
    h = sha256_file(p)
    print(f"PTQ calibration: {len(df)} eyes ({2*len(df)} images), full training split, "
          f"shuffle=False, sha256={h[:16]}...")
    return p, h, len(df)

CALIB_MANIFEST, CALIB_SHA, CALIB_N = calibration_manifest()
def calib_loader(): return loader(DualViewDataset(TRAIN_CSV), 16, False)   # shuffle=False

def run_ptq(fp32_ckpt, max_batches=None):
    max_batches = max_batches if max_batches is not None else (4 if QUICK else 10 ** 9)
    ref = fresh_student(fp32_ckpt)
    m = fresh_student(fp32_ckpt); m.eval()
    m.backbone.fuse_model(qat=False)
    m.backbone = QuantizableBackbone(m.backbone)
    m.backbone.qconfig = get_default_qconfig(QUANT_ENGINE)
    prep = prepare(m, inplace=False)
    n = 0
    with torch.no_grad():
        for i, b in enumerate(calib_loader()):
            prep(b["macula"], b["disc"]); n = i + 1
            if n >= max_batches: break
    q = convert(prep, inplace=False)
    if count_quantized(q) == 0: raise RuntimeError("PTQ produced no quantized modules")
    return q, {"path": "eager_static_ptq", "scope": QUANT_SCOPE, "engine": QUANT_ENGINE,
               "calib_batches": n, "calib_eyes": CALIB_N, "calib_sha256": CALIB_SHA,
               "quantized_ops": count_quantized(q), "eligible_ops": eligible_backbone_ops(ref),
               "quantized_module_paths": quantized_module_paths(q)}

def build_qat_prepared(fp32_ckpt):
    m = Student(init=INIT_TH)
    m.load_state_dict(load_weights(fp32_ckpt, "cpu"))
    m = m.to("cpu").train()
    m.backbone.fuse_model(qat=True)
    m.backbone = QuantizableBackbone(m.backbone)
    m.backbone.qconfig = get_default_qat_qconfig(QUANT_ENGINE)
    prepare_qat(m, inplace=True)
    return m

def count_fake_quant(m):
    return sum(1 for x in m.modules() if "fakequant" in type(x).__name__.lower())

def count_active_fake_quant(m):
    n = 0
    for x in m.modules():
        if hasattr(x, "fake_quant_enabled"):
            try: n += int(x.fake_quant_enabled.sum().item() > 0)
            except Exception: n += 1
    return n

def finetune(model, tag, lr, seed, cfg, eval_converted=False):
    """Shared loop for QAT and its FP32 control, so the ONLY difference is whether fake quantization
    is active. Objective is task supervision only -- no KD, no CSD."""
    set_seed(seed)
    pw = POS_W.to(DEVICE)
    tl = loader(DualViewDataset(TRAIN_CSV, paired=paired_tf), cfg["batch_size"], True, seed)
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    stop, hist = EarlyStopping(cfg["patience"]), []
    for ep in range(cfg["epochs"]):
        model.train()
        for b in tqdm(tl, desc=f"[{tag}] ep{ep}", leave=False):
            m, d, y = b["macula"].to(DEVICE), b["disc"].to(DEVICE), b["label"].to(DEVICE)
            o = model(m, d)
            loss = coral_loss(o["logit_dual"], y, pw) + 0.5 * aux_loss(o, y, pw)
            opt.zero_grad(); loss.backward(); opt.step()
        q_fake = quick_qwk(model, VAL_LOADER)
        q = q_fake
        if eval_converted:
            try:
                cm = convert(copy.deepcopy(model).to("cpu").eval(), inplace=False)
                r = predict_cpu(cm, VAL_LOADER); q = fast_qwk(r["y_true"], r["y_pred"]); del cm
            except Exception as e:
                print(f"  {tag}: converted eval failed ({e!r}); using fake-quant QWK")
        hist.append({"epoch": ep, "val_qwk": q, "val_qwk_fakequant": q_fake})
        print(f"  {tag} ep{ep}: val_QWK={q:.4f}" +
              (f" (converted INT8; fake-quant {q_fake:.4f})" if eval_converted else ""))
        stopped = stop.step(q, model, ep)
        model = model.to(DEVICE)
        if stopped: break
    stop.restore(model)
    pd.DataFrame(hist).to_csv(f"{LOG_DIR}/{tag}.csv", index=False)
    return model, stop.best, len(hist)

def run_qat(fp32_ckpt, lr, seed, cfg=None, tag_extra=""):
    cfg = cfg or QAT_CFG
    model = build_qat_prepared(fp32_ckpt)
    n_fq = count_fake_quant(model)
    # Assert fake quantization is active DURING training -- counting quantized modules after
    # conversion proves nothing about what happened while training.
    assert n_fq > 0, "QAT prepare produced no FakeQuantize modules"
    model, best, n_ep = finetune(model, f"QAT_s{seed}{tag_extra}", lr, seed, cfg, eval_converted=True)
    q = convert(model.to("cpu").eval(), inplace=False)
    return q, {"path": "eager_qat", "scope": QUANT_SCOPE, "engine": QUANT_ENGINE, "lr": lr,
               "seed": seed, "best_val_qwk": float(best), "epochs_run": n_ep,
               "selected_on": "converted_int8_validation_qwk",
               "fake_quant_during_training": n_fq,
               "quantized_ops": count_quantized(q), "eligible_ops": eligible_backbone_ops(fresh_student(fp32_ckpt)),
               "quantized_module_paths": quantized_module_paths(q)}

def run_fp32_ft_control(fp32_ckpt, lr, seed, cfg=None):
    """PRIMARY QAT control: the SAME fused, QAT-prepared graph with fake quantization and observers
    DISABLED. An ordinary unfused fine-tune would differ in graph structure too, and so could not
    isolate the one variable it is named after."""
    cfg = cfg or QAT_CFG
    m = build_qat_prepared(fp32_ckpt)
    m.apply(disable_fake_quant); m.apply(disable_observer)
    assert count_active_fake_quant(m) == 0, "control still has active fake quantization"
    m, best, _ = finetune(m, f"FP32FT_s{seed}", lr, seed, cfg, eval_converted=False)
    m.apply(disable_fake_quant); m.apply(disable_observer)
    return m.eval(), float(best)

def run_fp32_ft_plain(fp32_ckpt, lr, seed, cfg=None):
    """Ordinary FP32 fine-tune of the plain student. FT-PTQ is built from THIS, so no weight surgery
    is involved: fine-tuning and quantization happen on the same module tree PTQ uses."""
    cfg = cfg or QAT_CFG
    out = f"{CKPT_DIR}/student/fp32_ft_plain/seed{seed}.pt"
    if os.path.exists(out): return out
    m, best, _ = finetune(fresh_student(fp32_ckpt), f"FP32FTplain_s{seed}", lr, seed, cfg)
    torch_save({"model_state": m.state_dict(), "val_qwk": float(best), "seed": seed}, out)
    return out

print("Quantization helpers defined.")

In [ ]:
# ---- QAT learning rate, chosen on VALIDATION over the tuning seeds ----
_rows, _invalid = [], []
for lr in QAT_CFG["lr_grid"]:
    qs = []
    for sd in SEEDS_TUNING:
        try:
            _m, _i = run_qat(RQ2_BASE[sd], lr, sd, tag_extra=f"_lr{lr:g}")
            if np.isfinite(_i["best_val_qwk"]): qs.append(_i["best_val_qwk"])
            del _m
        except Exception as e:
            print(f"  QAT lr={lr:g} seed={sd} FAILED: {e!r}")
    if len(qs) != len(SEEDS_TUNING):
        _invalid.append({"lr": lr, "n_ok": len(qs)}); print(f"  !! QAT lr={lr:g} INVALID"); continue
    _rows.append({"lr": lr, "n_seeds": len(qs), "val_QWK_mean": float(np.mean(qs)),
                  "val_QWK_sd": float(np.std(qs, ddof=1)) if len(qs) > 1 else 0.0})
QAT_GRID = pd.DataFrame(_rows).sort_values("val_QWK_mean", ascending=False).reset_index(drop=True)
QAT_GRID.to_csv(f"{TAB_DIR}/table_01_grid_qat_lr.csv", index=False)
gate("Gate5_Grid_qat_lr", len(_rows) == len(QAT_CFG["lr_grid"]),
     f"{len(_rows)}/{len(QAT_CFG['lr_grid'])} learning rates completed all tuning seeds",
     blocking=BLOCK)
if not len(QAT_GRID): raise GateFailure("no QAT learning rate completed every tuning seed")
QAT_LR = float(QAT_GRID.iloc[0]["lr"])
print(QAT_GRID.round(4).to_string(index=False))
print(f"selected QAT lr = {QAT_LR:g} on mean CONVERTED-INT8 validation QWK "
      "(the FP32 control uses the same value)")

In [ ]:
# ---- one matched variant set per core seed ----
PTQ_M, PTQ_I = {}, {}
QAT_M, QAT_I = {}, {}
FT_M, FT_VAL = {}, {}
PLAIN_CKPT, FTPTQ_M, FTPTQ_I = {}, {}, {}
RQ2_SEEDS = sorted(RQ2_BASE)

for sd in RQ2_SEEDS:
    base = RQ2_BASE[sd]
    print(f"\n===== RQ2 seed {sd} =====")
    try:
        PTQ_M[sd], PTQ_I[sd] = run_ptq(base)
        print(f"  PTQ_{sd}: {PTQ_I[sd]['quantized_ops']} quantized ops, "
              f"{PTQ_I[sd]['calib_batches']} calibration batches")
    except Exception as e: print(f"  PTQ seed {sd} FAILED: {e!r}")
    try:
        FT_M[sd], FT_VAL[sd] = run_fp32_ft_control(base, QAT_LR, sd)
    except Exception as e: print(f"  FP32-FT control seed {sd} FAILED: {e!r}")
    try:
        PLAIN_CKPT[sd] = run_fp32_ft_plain(base, QAT_LR, sd)
        FTPTQ_M[sd], FTPTQ_I[sd] = run_ptq(PLAIN_CKPT[sd])
        print(f"  FT-PTQ_{sd}: {FTPTQ_I[sd]['quantized_ops']} quantized ops")
    except Exception as e: print(f"  FT-PTQ seed {sd} FAILED: {e!r}")
    try:
        QAT_M[sd], QAT_I[sd] = run_qat(base, QAT_LR, sd)
    except Exception as e: print(f"  QAT seed {sd} FAILED: {e!r}")

# ---- integrity ----
counts = {"best_fp32": len(RQ2_BASE), "ptq_int8": len(PTQ_M), "qat_int8": len(QAT_M),
          "fp32_ft_control": len(FT_M), "ft_ptq_int8": len(FTPTQ_M)}
short = {k: v for k, v in counts.items() if v != len(RQ2_SEEDS)}
gate("Gate7a_RQ2Completeness", not short,
     f"per-variant seed counts {counts} (required {len(RQ2_SEEDS)} each)"
     + ("" if not short else f"; INCOMPLETE {short}"), blocking=BLOCK)

# EXACT operator-set equality, PER SEED -- counting cannot show the same operators were quantized,
# and a representative pair cannot show it held for the other seeds the pairing relies on.
mismatch = []
for sd in RQ2_SEEDS:
    sets = {n: set(d[sd]["quantized_module_paths"]) for n, d in
            (("ptq", PTQ_I), ("qat", QAT_I), ("ft_ptq", FTPTQ_I)) if sd in d}
    if len(sets) < 2: mismatch.append(f"seed {sd}: only {sorted(sets)}"); continue
    ref_n, ref = next(iter(sets.items()))
    for n, s_ in sets.items():
        if s_ != ref: mismatch.append(f"seed {sd}: {n} != {ref_n}")
save_json({str(sd): {n: d[sd]["quantized_module_paths"] for n, d in
                     (("ptq", PTQ_I), ("qat", QAT_I), ("ft_ptq", FTPTQ_I)) if sd in d}
           for sd in RQ2_SEEDS}, f"{CFG_DIR}/quantized_modules_by_seed.json")
gate("Gate7b_QuantScopeMatched", not mismatch,
     f"operator sets identical across PTQ/QAT/FT-PTQ for all {len(RQ2_SEEDS)} seeds"
     if not mismatch else f"MISMATCH {mismatch[:3]}", blocking=BLOCK)

_ptq_ok = bool(PTQ_M) and all(count_quantized(m) > 0 for m in PTQ_M.values())
_qat_ok = (len(QAT_M) == len(RQ2_SEEDS) and all(count_quantized(m) > 0 for m in QAT_M.values())
           and all(QAT_I[s]["fake_quant_during_training"] > 0 for s in QAT_M))
gate("Gate7c_PTQ_Integrity", _ptq_ok,
     f"{len(PTQ_M)}/{len(RQ2_SEEDS)} seeds; scope={QUANT_SCOPE}; "
     f"{PTQ_I[RQ2_SEEDS[0]]['quantized_ops']}/{PTQ_I[RQ2_SEEDS[0]]['eligible_ops']} backbone ops",
     blocking=BLOCK)
gate("Gate7d_QAT_Integrity", _qat_ok,
     f"{len(QAT_M)}/{len(RQ2_SEEDS)} seeds; lr={QAT_LR:g}; fake quantization active during training",
     blocking=BLOCK)

save_json({"scope": QUANT_SCOPE, "engine": QUANT_ENGINE, "qat_lr": QAT_LR,
           "rq2_seeds": RQ2_SEEDS, "rq2_base_condition": BEST_CONDITION,
           "design": "per-seed matched: FP32_s -> {PTQ_s, QAT_s, FP32FT_s, FT-PTQ_s}",
           "calibration": {"eyes": CALIB_N, "shuffle": False, "sha256": CALIB_SHA},
           "artifact": "INT8 backbone with FP32 fusion and ordinal heads (mixed precision)",
           "ptq": {str(k): {kk: vv for kk, vv in v.items() if kk != "quantized_module_paths"}
                   for k, v in PTQ_I.items()},
           "qat": {str(k): {kk: vv for kk, vv in v.items() if kk != "quantized_module_paths"}
                   for k, v in QAT_I.items()}}, f"{CFG_DIR}/quantization_info.json")

## 16 — RQ2 on validation, and the frozen deployment decision

**This is the last decision before the test set is touched.** The order is the only defensible one:

```
validation → deployment selection → FREEZE → DRTiD test → DeepDRiD
```

The severe-error clause is evaluated on **severe error** (the CI of `ΔSER = SER_int8 − SER_fp32`),
not on QWK, and the rule is **fail-closed**: a candidate with missing or non-finite evidence is
rejected, never assumed safe. `min_qwk_retention_pct` is a pre-specified *engineering* criterion for
choosing which artifact to ship — not a clinical non-inferiority margin.

In [ ]:
VAL_PRED, _rq2_val = {}, []
QUANT_OF = {"ptq_int8": "PTQ_INT8", "qat_int8": "QAT_INT8", "ft_ptq_int8": "FT_PTQ_INT8"}

def score_val(cond, seed, model, quant, on_cpu, ckpt=""):
    if on_cpu:
        r = predict_cpu(model, VAL_LOADER)
        yt, yp, pc, cid = r["y_true"], r["y_pred"], r["p_dual"], r["cluster_ids"]
    else:
        yt, yp, pc, cid = predict(model, VAL_LOADER, clusters=True)
    m = all_metrics(yt, yp, pc)
    VAL_PRED[(cond, seed, quant)] = {"y_true": yt, "y_pred": yp, "cluster_ids": cid}
    lat = benchmark_latency(model, _sample["macula"], _sample["disc"])
    _, sz = state_dict_size_mb(model, f"val_{cond}_s{seed}")
    _rq2_val.append({"condition": cond, "seed": seed, "quantization": quant,
                     "val_QWK": m["QWK"], "val_MacroF1": m["MacroF1"], "val_Accuracy": m["Accuracy"],
                     "val_MAE": m["MAE"], "val_SevereErrorRate": m["SevereErrorRate"],
                     "Latency_median_ms": lat["Latency_median_ms"], "CheckpointSize_MB": sz,
                     "checkpoint": ckpt})

_sample = next(iter(VAL_LOADER))
for sd in RQ2_SEEDS:
    _m = load_student(RQ2_BASE[sd]); score_val("best_fp32", sd, _m, "FP32", False, RQ2_BASE[sd]); del _m
for sd, m in FT_M.items():          score_val("fp32_ft_control", sd, m.to(DEVICE), "FP32", False)
for sd, ck in PLAIN_CKPT.items():
    _m = load_student(ck); score_val("fp32_ft_plain", sd, _m, "FP32", False, ck); del _m
for sd, m in PTQ_M.items():         score_val("ptq_int8", sd, m, "PTQ_INT8", True)
for sd, m in FTPTQ_M.items():       score_val("ft_ptq_int8", sd, m, "FT_PTQ_INT8", True)
for sd, m in QAT_M.items():         score_val("qat_int8", sd, m, "QAT_INT8", True)

RQ2_VAL = pd.DataFrame(_rq2_val)
RQ2_VAL.to_csv(f"{TAB_DIR}/table_04a_rq2_validation_per_seed.csv", index=False)
RQ2_VAL_SUM = (RQ2_VAL.groupby("condition")
               .agg(n_seeds=("seed", "size"), val_QWK=("val_QWK", "mean"),
                    val_QWK_sd=("val_QWK", "std"), val_MacroF1=("val_MacroF1", "mean"),
                    val_SER=("val_SevereErrorRate", "mean"), val_MAE=("val_MAE", "mean"),
                    Latency_median_ms=("Latency_median_ms", "mean"),
                    CheckpointSize_MB=("CheckpointSize_MB", "mean")).reset_index())
RQ2_VAL_SUM.to_csv(f"{TAB_DIR}/table_04_rq2_validation.csv", index=False)
print("RQ2 on VALIDATION (the only data the deployment decision may use):")
print(RQ2_VAL_SUM.round(4).to_string(index=False))

In [ ]:
def metric_fns():
    return {"QWK": lambda t, p: fast_qwk(t, p),
            "Accuracy": lambda t, p: float((t == p).mean()),
            "MacroF1": lambda t, p: float(f1_score(t, p, average="macro",
                                                   labels=list(range(NUM_CLASSES)), zero_division=0)),
            "MAE": lambda t, p: float(np.mean(np.abs(t - p))),
            "SevereErrorRate": lambda t, p: float(np.mean(np.abs(t - p) >= 2))}

def paired_delta_val(cond, metric="SevereErrorRate", B=None, rng_seed=7):
    """Cluster + seed bootstrap of metric(INT8_s) - metric(FP32_s) on VALIDATION."""
    B = B or (300 if QUICK else BOOTSTRAP_B)
    fn = metric_fns()[metric]; q = QUANT_OF.get(cond, "FP32")
    seeds = [s for s in RQ2_SEEDS if (cond, s, q) in VAL_PRED and ("best_fp32", s, "FP32") in VAL_PRED]
    if not seeds: return None
    ref = VAL_PRED[("best_fp32", seeds[0], "FP32")]
    yt, cl = ref["y_true"], ref["cluster_ids"]
    uniq = np.unique(cl); idx = {c: np.where(cl == c)[0] for c in uniq}
    rng = np.random.default_rng(rng_seed); d = np.empty(B)
    for b in range(B):
        pick = np.concatenate([idx[c] for c in rng.choice(uniq, len(uniq), replace=True)])
        sel = rng.integers(0, len(seeds), len(seeds))
        d[b] = float(np.mean([fn(yt[pick], VAL_PRED[(cond, seeds[j], q)]["y_pred"][pick])
                              - fn(yt[pick], VAL_PRED[("best_fp32", seeds[j], "FP32")]["y_pred"][pick])
                              for j in sel]))
    lo, hi = np.percentile(d, [2.5, 97.5])
    return {"condition": cond, "metric": metric, "n_seeds": len(seeds),
            "mean_diff": float(d.mean()), "ci_low": float(lo), "ci_high": float(hi),
            "credibly_worse": bool(lo > 0)}

VAL_DELTAS = pd.DataFrame([r for c in ("ptq_int8", "ft_ptq_int8", "qat_int8")
                           for met in ("SevereErrorRate", "QWK")
                           for r in [paired_delta_val(c, met)] if r])
if len(VAL_DELTAS):
    VAL_DELTAS.to_csv(f"{TAB_DIR}/table_04b_rq2_validation_deltas.csv", index=False)
    print("\nPaired validation deltas vs FP32:")
    print(VAL_DELTAS.round(4).to_string(index=False))

def choose_deployment():
    """Fail-closed: missing or non-finite evidence REJECTS a candidate, never approves it."""
    base = RQ2_VAL_SUM[RQ2_VAL_SUM.condition == "best_fp32"]
    if not len(base): return "best_fp32", "no FP32 reference -- fail-closed"
    q0 = float(base["val_QWK"].iloc[0])
    if not (np.isfinite(q0) and q0 > 0): return "best_fp32", f"FP32 reference unusable ({q0})"
    integrity = {"ptq_int8": _ptq_ok, "ft_ptq_int8": bool(FTPTQ_M), "qat_int8": _qat_ok}
    n_by = {"ptq_int8": len(PTQ_M), "ft_ptq_int8": len(FTPTQ_M), "qat_int8": len(QAT_M)}
    cands, audit = [], []
    for c in ("ptq_int8", "ft_ptq_int8", "qat_int8"):
        sub = RQ2_VAL_SUM[RQ2_VAL_SUM.condition == c]
        why = []
        if not len(sub): why.append("no validation row")
        qv = float(sub["val_QWK"].iloc[0]) if len(sub) else float("nan")
        lat = float(sub["Latency_median_ms"].iloc[0]) if len(sub) else float("nan")
        ret = 100.0 * qv / q0 if np.isfinite(qv) else float("nan")
        if not np.isfinite(qv):  why.append("validation QWK not finite")
        if not np.isfinite(lat): why.append("latency not measured")
        if not integrity.get(c): why.append("quantization integrity not PASS")
        if n_by.get(c, 0) != len(RQ2_SEEDS): why.append(f"only {n_by.get(c,0)}/{len(RQ2_SEEDS)} seeds")
        d = (VAL_DELTAS[(VAL_DELTAS.condition == c) & (VAL_DELTAS.metric == "SevereErrorRate")]
             if len(VAL_DELTAS) else VAL_DELTAS)
        if not len(d): why.append("no severe-error delta (missing evidence rejects)")
        elif not (np.isfinite(d.iloc[0]["ci_low"]) and np.isfinite(d.iloc[0]["ci_high"])):
            why.append("severe-error CI not finite")
        elif bool(d.iloc[0]["credibly_worse"]): why.append("severe error credibly worse")
        if np.isfinite(ret) and ret < DEPLOY_RULE["min_qwk_retention_pct"]:
            why.append(f"retention {ret:.1f}% < {DEPLOY_RULE['min_qwk_retention_pct']}%")
        ok = not why
        audit.append({"condition": c, "val_QWK": qv, "retention_pct": ret,
                      "latency_median_ms": lat, "eligible": ok, "reasons": "; ".join(why)})
        print(f"  {c:14s} retention {ret:6.1f}%  -> " + ("ELIGIBLE" if ok else f"rejected ({'; '.join(why)})"))
        if ok: cands.append((lat, c, ret))
    pd.DataFrame(audit).to_csv(f"{TAB_DIR}/table_04c_deployment_eligibility.csv", index=False)
    if not cands:
        return "best_fp32", "no INT8 variant satisfied every criterion with complete evidence"
    cands.sort()
    return cands[0][1], (f"validation QWK retention {cands[0][2]:.1f}% (engineering criterion), "
                         f"severe error not credibly worse, lowest CPU latency {cands[0][0]:.2f} ms")

def representative_seed(cond):
    """Same rule for every variant -- QWK up, Macro-F1 up, severe error down, MAE down."""
    sub = RQ2_VAL[RQ2_VAL.condition == cond]
    if not len(sub): return BEST_SEED
    return int(sub.sort_values(["val_QWK", "val_MacroF1", "val_SevereErrorRate", "val_MAE"],
                               ascending=[False, False, True, True]).iloc[0]["seed"])

REP_SEED = {c: representative_seed(c) for c in RQ2_VAL.condition.unique()}
print("\nRepresentative seed per variant (identical validation rule):", REP_SEED)

print("\nApplying the pre-registered deployment rule to VALIDATION results:")
DEPLOY_CHOICE, DEPLOY_REASON = choose_deployment()
DEPLOY_SEED = REP_SEED.get(DEPLOY_CHOICE, BEST_SEED)
DEPLOY_QUANT = QUANT_OF.get(DEPLOY_CHOICE, "FP32")
DEPLOY_FROZEN = True
print(f"\nDEPLOYMENT MODEL = {DEPLOY_CHOICE} (seed {DEPLOY_SEED}) -- {DEPLOY_REASON}")
print("FROZEN. Nothing after this line may change it.")
save_json({"chosen": DEPLOY_CHOICE, "seed": DEPLOY_SEED, "quantization": DEPLOY_QUANT,
           "reason": DEPLOY_REASON, "rule": DEPLOY_RULE, "decided_on": "DRTiD validation",
           "representative_seeds": {k: int(v) for k, v in REP_SEED.items()}},
          f"{RESULTS_DIR}/deployment_choice.json")

## 17 — Internal test evaluation

Every condition × seed is evaluated once on the held-out DRTiD test set. Per-sample predictions are
written to `predictions/`, so any future metric can be recomputed without re-running inference.

**Scope statement for the paper:** within this run the test set is not consulted for any selection
decision. Test findings below are *diagnostics*, reported in full and never allowed to change the
experiment.

In [ ]:
TEST_DS = DualViewDataset(TEST_CSV)
TEST_LOADER = loader(TEST_DS, 16, False)
TEACHER = get_teacher()
PRED_STORE, rows, warn_rows = {}, [], []

def save_predictions(cond, seed, yt, yp, pc, cid, quant, dataset="DRTiD", split="test"):
    df = pd.DataFrame({"dataset": dataset, "split": split, "cluster_id": cid,
                       "cluster_level": "record_eye" if dataset == "DRTiD" else "patient",
                       "sample_id": [f"{dataset}_{c}" for c in cid] if dataset == "DRTiD"
                                    else [f"{dataset}_{c}_{i}" for i, c in enumerate(cid)],
                       "true_grade": yt, "pred_grade": yp,
                       "condition": cond, "seed": seed, "quantization": quant})
    for k in range(pc.shape[1]): df[f"p_threshold_{k}"] = pc[:, k].cpu().numpy()
    df.to_csv(f"{PRED_DIR}/{dataset}_{split}_{cond}_seed{seed}_{quant}.csv", index=False)

def evaluate(model, cond, seed, view="dual", quant="FP32", on_cpu=False, ld=None,
             with_shift=True, with_mem=False, fp32_ref=None, tag=None):
    ld = ld or TEST_LOADER
    if on_cpu:
        r = predict_cpu(model, ld)
        yt, yp, pc, cid = r["y_true"], r["y_pred"], r["p_dual"], r["cluster_ids"]
    else:
        yt, yp, pc, cid = predict(model, ld, view=view, clusters=True)
    m = all_metrics(yt, yp, pc)
    row = {"condition": cond, "seed": seed, "quantization": quant, "view_mode": view, **m}
    if view == "dual":
        if on_cpu:
            qd, qm, qdd = fast_qwk(yt, yp), fast_qwk(yt, r["y_pred_macula"]), fast_qwk(yt, r["y_pred_disc"])
            row.update({"QWK_dual": qd, "QWK_aux_macula": qm, "QWK_aux_disc": qdd,
                        "DualViewGain_G_aux": qd - max(qm, qdd)})
        else:
            row.update(dual_view_gain(model, ld))
            if with_shift:
                row.update(shift_fidelity(TEACHER, model, ld))
                row.update(shift_fidelity(TEACHER, model, ld, cf=True))
    # Quantization does not change the ARCHITECTURAL parameter count -- packed INT8 weights simply
    # stop appearing as Parameters. Size carries the compression story.
    row["ParamCount"] = param_count(fp32_ref if fp32_ref is not None else model)
    _, sz = state_dict_size_mb(model, tag or f"{cond}_s{seed}_{quant}")
    row["CheckpointSize_MB"] = sz
    row.update(benchmark_latency(model, _sample["macula"], _sample["disc"]))
    if with_mem:
        row.update(measure_memory(lambda: copy.deepcopy(model).to("cpu").eval(),
                                  _sample["macula"], _sample["disc"]))
    w = collapse_warnings(yt, yp)
    if w:
        warn_rows.append({"condition": cond, "seed": seed, "quantization": quant, "warnings": "; ".join(w)})
        print(f"  !! {cond}|s{seed}|{quant}: " + "; ".join(w))
    save_predictions(cond, seed, yt, yp, pc, cid, quant)
    cm = confusion_matrix(yt, yp, labels=list(range(NUM_CLASSES)))
    pd.DataFrame(cm, index=[f"true_{g}" for g in range(NUM_CLASSES)],
                 columns=[f"pred_{g}" for g in range(NUM_CLASSES)]).to_csv(
        f"{MET_DIR}/confusion_{cond}_seed{seed}_{quant}.csv")
    PRED_STORE[(cond, seed, quant)] = {"y_true": yt, "y_pred": yp, "cluster_ids": cid}
    rows.append(row)
    return row

evaluate(TEACHER, "teacher", "-", with_shift=False)
for cond, view in (("macula_only", "macula_only"), ("disc_only", "disc_only")):
    for s in SEEDS_CORE:
        ck = f"{CKPT_DIR}/student/{cond}/seed{s}.pt"
        if os.path.exists(ck):
            m = load_student(ck); evaluate(m, cond, s, view=view, with_shift=False); del m
for cond in CORE_CONDITIONS:
    for s in SEEDS_CORE:
        ck = f"{CKPT_DIR}/student/{cond}/seed{s}.pt"
        if os.path.exists(ck):
            m = load_student(ck); evaluate(m, cond, s); del m
for cond in ("abl_csd_raw_smoothl1", "abl_csd_kl_softmax", "abl_csd_counterfactual"):
    for s in SEEDS_ABL:
        ck = f"{CKPT_DIR}/student/{cond}/seed{s}.pt"
        if os.path.exists(ck):
            m = load_student(ck); evaluate(m, cond, s); del m
print(f"\nEvaluated {len(rows)} model-runs so far.")

In [ ]:
# ---- RQ2 variants on the test set ----
BEST_FP32_MODEL = load_student(BEST_CKPT)
for sd in RQ2_SEEDS:
    m = load_student(RQ2_BASE[sd])
    evaluate(m, "best_fp32", sd, with_mem=(sd == BEST_SEED), tag=f"best_fp32_s{sd}"); del m
for sd, m in FT_M.items():
    evaluate(m.to(DEVICE), "fp32_ft_control", sd, with_shift=False, tag=f"fp32ft_s{sd}")
for sd, ck in PLAIN_CKPT.items():
    m = load_student(ck); evaluate(m, "fp32_ft_plain", sd, with_shift=False, tag=f"plain_s{sd}"); del m
for sd, m in PTQ_M.items():
    evaluate(m, "ptq_int8", sd, quant="PTQ_INT8", on_cpu=True, fp32_ref=BEST_FP32_MODEL,
             with_mem=(sd == BEST_SEED), tag=f"ptq_s{sd}")
for sd, m in FTPTQ_M.items():
    evaluate(m, "ft_ptq_int8", sd, quant="FT_PTQ_INT8", on_cpu=True, fp32_ref=BEST_FP32_MODEL,
             tag=f"ftptq_s{sd}")
for sd, m in QAT_M.items():
    evaluate(m, "qat_int8", sd, quant="QAT_INT8", on_cpu=True, fp32_ref=BEST_FP32_MODEL,
             with_mem=(sd == REP_SEED.get("qat_int8")), tag=f"qat_s{sd}")

RAW = pd.DataFrame(rows)

# G_independent: dual-view vs INDEPENDENTLY trained single-view students, matched WITHIN a seed.
_sv = RAW[RAW.condition.isin(["macula_only", "disc_only"])]
_ref_by_seed = {s: float(g["QWK"].max()) for s, g in _sv.groupby("seed")} if len(_sv) else {}
_ref_mean = float(_sv["QWK"].max()) if len(_sv) else float("nan")
RAW["DualViewGain_G_independent"] = RAW.apply(
    lambda r: r["QWK"] - _ref_by_seed.get(r["seed"], _ref_mean) if r["view_mode"] == "dual" else np.nan,
    axis=1)
RAW.to_csv(f"{MET_DIR}/all_conditions_raw.csv", index=False)
if warn_rows: pd.DataFrame(warn_rows).to_csv(f"{MET_DIR}/collapse_warnings.csv", index=False)
print("G_independent reference (best single-view QWK) by seed:",
      {k: round(v, 4) for k, v in _ref_by_seed.items()})

# ---- completeness of the required columns ----
REQ1 = ["QWK", "MacroF1", "ShiftL1", "CosAgree", "BenefitCorr"]
REQ2 = ["QWK", "Accuracy", "MacroF1", "CheckpointSize_MB", "Latency_median_ms", "DualViewGain_G_aux"]
EXPECT2 = ["best_fp32", "fp32_ft_control", "ft_ptq_int8", "ptq_int8", "qat_int8"]
present = set(RAW.condition)
miss_cond = {"RQ1": [c for c in CORE_CONDITIONS if c not in present],
             "RQ2": [c for c in EXPECT2 if c not in present]}
def miss_cols(conds, req):
    out = {}
    for c in conds:
        if c not in present: continue
        s = RAW[RAW.condition == c]
        for col in req:
            if col not in s.columns or s[col].isna().all(): out.setdefault(c, []).append(col)
    return out
m1, m2 = miss_cols(CORE_CONDITIONS, REQ1), miss_cols(EXPECT2, REQ2)
gate("Gate8_RQ_Completeness", not (m1 or m2 or miss_cond["RQ1"] or miss_cond["RQ2"]),
     f"missing conditions={miss_cond} | RQ1 columns={m1 or 'none'} | RQ2 columns={m2 or 'none'}",
     blocking=BLOCK)

# Test-set findings are DIAGNOSTIC. Viability was already gated on validation; blocking here would
# let test performance decide whether the experiment may continue.
gate("Gate9_TestDiagnostics", True,
     f"{len(warn_rows)} condition(s) flagged for rare-grade misses or collapse -- reported as "
     "findings, non-blocking by design")

agg = RAW.groupby("condition")[["QWK", "Accuracy", "MacroF1", "MAE", "SevereErrorRate"]].agg(["mean", "std"])
print(agg.round(4).to_string())

## 18 — Statistics

**Hierarchical paired cluster bootstrap over matched seeds.** Each replicate resamples eye-level
clusters with replacement, resamples the seed PAIRS with replacement, and averages the paired
per-seed difference:

```
Δ_b = (1/S) Σ_s [ M(A_s, b) − M(B_s, b) ]        with seed s the SAME on both sides
```

The point estimate is the **observed** paired difference on the real data; the bootstrap supplies
the interval. p-values come from a paired cluster permutation test, Holm-corrected **within** each
pre-registered family (RQ1: 3 hypotheses, RQ2: 5).

Clustering is at DRTiD's **record/eye** level — its public metadata exposes no patient key, so this
is never described as patient-clustered.

**Effect size with a CI is the headline; p-values are secondary.**

In [ ]:
def stack_preds(cond, seeds, quant="FP32"):
    yt = cl = None; preds = {}
    for s in seeds:
        rec = PRED_STORE.get((cond, s, quant))
        if rec is None: continue
        if yt is None: yt, cl = rec["y_true"], rec["cluster_ids"]
        elif not np.array_equal(yt, rec["y_true"]):
            raise AssertionError(f"{cond} seed {s} has a different sample order -- pairing invalid")
        preds[s] = rec["y_pred"]
    return yt, cl, preds

def seed_pairs(pa, pb):
    """Primary comparisons require EXACT matched seed sets. Silently pairing seed 42 against 8888
    would report an unpaired difference as paired."""
    if set(pa) != set(pb):
        raise RuntimeError(f"paired comparison needs matched seeds, got {sorted(pa)} vs {sorted(pb)}")
    return [(s, s) for s in sorted(pa)]

def paired_bootstrap(a, b, seeds_a, seeds_b, B=None, qa="FP32", qb="FP32", rng_seed=0):
    B = B or BOOTSTRAP_B
    yta, cl, pa = stack_preds(a, seeds_a, qa)
    ytb, _, pb = stack_preds(b, seeds_b, qb)
    if yta is None or ytb is None or not pa or not pb: return None
    if not np.array_equal(yta, ytb): raise AssertionError("sample order differs across conditions")
    pairs = seed_pairs(pa, pb)
    fns = metric_fns()
    uniq = np.unique(cl); idx = {c: np.where(cl == c)[0] for c in uniq}
    rng = np.random.default_rng(rng_seed)
    draws = {k: np.empty(B) for k in fns}
    for i in range(B):
        pick = np.concatenate([idx[c] for c in rng.choice(uniq, len(uniq), replace=True)])
        sel = rng.integers(0, len(pairs), len(pairs))
        yt = yta[pick]
        for k, fn in fns.items():
            draws[k][i] = float(np.mean([fn(yt, pa[pairs[j][0]][pick]) - fn(yt, pb[pairs[j][1]][pick])
                                         for j in sel]))
    out = {}
    for k, fn in fns.items():
        obs = float(np.mean([fn(yta, pa[x]) - fn(yta, pb[y]) for x, y in pairs]))
        lo, hi = np.percentile(draws[k], [100 * ALPHA_CI / 2, 100 * (1 - ALPHA_CI / 2)])
        out[k] = {"observed_diff": obs, "bootstrap_mean": float(draws[k].mean()),
                  "ci_low": float(lo), "ci_high": float(hi),
                  "excludes_zero": bool(lo > 0 or hi < 0), "n_seed_pairs": len(pairs)}
    return out

def permutation_test(a, b, seeds_a, seeds_b, metric="QWK", P=None, qa="FP32", qb="FP32", rng_seed=0):
    """Exchange the two methods' predictions within each cluster at random. Null = the method label
    is exchangeable. Two-sided p with the +1 correction, so p is never exactly 0."""
    P = P or PERMUTATION_P
    yt, cl, pa = stack_preds(a, seeds_a, qa)
    _, _, pb = stack_preds(b, seeds_b, qb)
    if yt is None or not pa or not pb: return None
    fn = metric_fns()[metric]
    pairs = [(pa[x], pb[y]) for x, y in seed_pairs(pa, pb)]
    obs = float(np.mean([fn(yt, xa) - fn(yt, xb) for xa, xb in pairs]))
    uniq = np.unique(cl); masks = {c: (cl == c) for c in uniq}
    rng = np.random.default_rng(rng_seed); cnt = 0
    for _ in range(P):
        swap = rng.random(len(uniq)) < 0.5
        sel = np.zeros(len(cl), bool)
        for c, s_ in zip(uniq, swap):
            if s_: sel |= masks[c]
        d = [fn(yt, np.where(sel, xb, xa)) - fn(yt, np.where(sel, xa, xb)) for xa, xb in pairs]
        if abs(np.mean(d)) >= abs(obs) - 1e-12: cnt += 1
    return {"observed_diff": obs, "p_perm": float((cnt + 1) / (P + 1)), "n_permutations": P}

def holm(pvals, names, alpha=0.05):
    m = len(pvals); order = np.argsort(pvals); adj = [None] * m; run = 0.0
    for rank, i in enumerate(order):
        run = max(run, min(1.0, (m - rank) * pvals[i])); adj[i] = run
    return {names[i]: {"p_raw": float(pvals[i]), "p_holm": float(adj[i]),
                       "significant_holm": bool(adj[i] < alpha)} for i in range(m)}

SEEDS_OF = {c: RQ2_SEEDS for c in ("best_fp32", "fp32_ft_control", "ft_ptq_int8",
                                   "ptq_int8", "qat_int8")}
stat_rows, pv, names, fams = [], [], [], []
for rq, pairs in PREREGISTERED.items():
    for a, b in pairs:
        sa, sb = SEEDS_OF.get(a, SEEDS_CORE), SEEDS_OF.get(b, SEEDS_CORE)
        qa, qb = QUANT_OF.get(a, "FP32"), QUANT_OF.get(b, "FP32")
        try:
            res = paired_bootstrap(a, b, sa, sb, qa=qa, qb=qb)
            perm = permutation_test(a, b, sa, sb, "QWK", qa=qa, qb=qb)
        except (AssertionError, RuntimeError) as e:
            print(f"  skip {rq} {a} vs {b}: {e}"); continue
        if res is None: print(f"  skip {rq} {a} vs {b}: predictions unavailable"); continue
        for met, r in res.items():
            row = {"RQ": rq, "comparison": f"{a}_vs_{b}", "metric": met,
                   "cluster_level": "record_eye(DRTiD)", **r}
            if met == "QWK" and perm:
                row.update({"p_perm": perm["p_perm"], "n_permutations": perm["n_permutations"]})
                pv.append(perm["p_perm"]); names.append(f"{rq}:{a}_vs_{b}"); fams.append(rq)
            stat_rows.append(row)
        q = res["QWK"]
        print(f"  {rq} {a} vs {b}: dQWK={q['observed_diff']:+.4f} "
              f"[{q['ci_low']:+.4f},{q['ci_high']:+.4f}]" + (f" p={perm['p_perm']:.4f}" if perm else ""))

STATS = pd.DataFrame(stat_rows)
if pv:
    # Holm WITHIN each pre-registered family: RQ1 and RQ2 are separate questions declared in
    # advance, so pooling them would penalise each for the other's existence.
    corr = {}
    for fam in sorted(set(fams)):
        ix = [i for i, f in enumerate(fams) if f == fam]
        for k, v in holm([pv[i] for i in ix], [names[i] for i in ix]).items():
            corr[k] = {**v, "family": fam, "family_size": len(ix)}
    key = STATS.apply(lambda r: f"{r['RQ']}:{r['comparison']}", axis=1)
    STATS["p_holm"] = [corr.get(k, {}).get("p_holm", np.nan) if m == "QWK" else np.nan
                       for k, m in zip(key, STATS["metric"])]
    STATS["significant_holm"] = [corr.get(k, {}).get("significant_holm", None) if m == "QWK" else None
                                 for k, m in zip(key, STATS["metric"])]
    print("Holm applied within each family: " +
          ", ".join(f"{f}={sum(1 for x in fams if x == f)}" for f in sorted(set(fams))))
STATS.to_csv(f"{TAB_DIR}/table_05_statistics.csv", index=False)
gate("Gate10_Statistics", len(STATS) > 0,
     f"{len(STATS)} comparisons (bootstrap B={BOOTSTRAP_B}, permutations={PERMUTATION_P}, "
     "Holm within family)")
print("A difference whose CI includes zero is NOT a claim, whatever the point estimate.")

## 19 — DeepDRiD external validation

Models are **frozen** here: no fine-tuning, no threshold tuning, no selection, no method change.
A drop versus DRTiD is a domain-shift finding to report, not something to engineer away.

**Set-C** (`Online-Challenge1&2-Evaluation`) is the partition the challenge reserved for final
evaluation, and nothing in this project has looked at it — so it is the **confirmatory** result.
Set-B and Set-A are supplementary external validation.

Patient IDs confirm the partitions are disjoint (Set-A 1–330, Set-B 265–433, Set-C 347–500, zero
overlap between any pair). `cluster_id` is the **patient** (two eyes per patient are not independent);
`sample_id` is the eye.

**Field ordering is pre-registered.** DeepDRiD's CSVs never state which of `_1`/`_2` is
macula-centred, so both orderings are evaluated — but `_1=macula` is the headline and the reverse is
a sensitivity analysis. Whichever scores higher must not be promoted after the fact.

In [ ]:
class SetCDataset(Dataset):
    """Set-C: labels are one row per IMAGE in Challenge1_labels.xlsx; patient and eye are parsed out
    of image_id (347_l1 -> patient 347, eye l, field 1). An eye must have exactly fields {1, 2}."""
    SUB = "Online-Challenge1&2-Evaluation"
    def __init__(self, root, transform=None, order=DEEPDRID_PRIMARY_ORDER):
        self.tf, self.order = transform or eval_tf, order
        base = f"{root}/{self.SUB}"; lab = f"{base}/Challenge1_labels.xlsx"
        self.available = os.path.exists(lab)
        recs, excl = [], []
        if not self.available:
            self.df, self.excl = pd.DataFrame(), pd.DataFrame(); return
        df = pd.read_excel(lab)
        gcol = "DR_Levels" if "DR_Levels" in df.columns else df.columns[-1]
        df["image_id"] = df["image_id"].astype(str)
        df["pid"]   = df["image_id"].str.split("_").str[0]
        df["eye"]   = df["image_id"].str.extract(r"_([lr])")[0]
        df["field"] = df["image_id"].str.extract(r"_[lr](\d+)$")[0]
        disk = {}
        for dp, _, fns in os.walk(f"{base}/Images"):
            for fn in fns:
                if fn.lower().endswith((".jpg", ".jpeg", ".png")):
                    disk.setdefault(os.path.splitext(fn)[0], os.path.join(dp, fn))
        for (pid, eye), g in df.groupby(["pid", "eye"]):
            def drop(reason, grade=None):
                excl.append({"patient_id": pid, "eye": eye, "reason": reason,
                             "n_views": len(g), "image_ids": ",".join(g["image_id"]), "grade": grade})
            fields = sorted(g["field"].dropna().astype(str))
            if set(fields) != {"1", "2"}:
                drop("MISSING_FIELD" if len(fields) < 2 else "UNEXPECTED_FIELD_COUNT"); continue
            lv = g[gcol].dropna()
            if lv.empty: drop("MISSING_GRADE"); continue
            if lv.nunique() > 1: drop("INCONSISTENT_GRADE", ";".join(map(str, lv.unique()))); continue
            grade = int(lv.iloc[0])
            if not 0 <= grade < NUM_CLASSES: drop("INVALID_GRADE", grade); continue
            ids = g.sort_values("field")["image_id"].tolist()
            paths = [disk.get(i) for i in ids]
            if any(p is None for p in paths): drop("PATH_NOT_FOUND", grade); continue
            recs.append({"patient_id": int(pid), "eye": eye, "f1": paths[0], "f2": paths[1],
                         "grade": grade})
        self.df, self.excl = pd.DataFrame(recs), pd.DataFrame(excl)
        if len(self.excl): print(f"  Set-C: {len(self.excl)} eye(s) excluded "
                                 f"{self.excl.reason.value_counts().to_dict()}")
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        a, b = (r["f1"], r["f2"]) if self.order == "_1=macula" else (r["f2"], r["f1"])
        return {"macula": self.tf(image=load_img(a))["image"],
                "disc": self.tf(image=load_img(b))["image"],
                "label": torch.tensor(int(r["grade"]), dtype=torch.long),
                "cluster_id": int(r["patient_id"]), "sample_id": f"{int(r['patient_id'])}_{r['eye']}"}

class SetABDataset(Dataset):
    """Set-A / Set-B: labels come from the partition CSV; grades live in the column named after the
    OPPOSITE eye for a couple of patients (77, 164), which is a known DeepDRiD quirk -- such eyes are
    excluded and audited rather than repaired by inference."""
    def __init__(self, root, subs, transform=None, order=DEEPDRID_PRIMARY_ORDER):
        self.tf, self.order = transform or eval_tf, order
        recs, excl = [], []
        for sub in subs:
            csv = f"{root}/{sub}/{sub}.csv"
            if not os.path.exists(csv): continue
            df = pd.read_csv(csv)
            disk = {}
            for dp, _, fns in os.walk(f"{root}/{sub}/Images"):
                for fn in fns:
                    if fn.lower().endswith((".jpg", ".jpeg", ".png")):
                        disk.setdefault(os.path.splitext(fn)[0], os.path.join(dp, fn))
            for pid, grp in df.groupby("patient_id"):
                for eye, col in (("l", "left_eye_DR_Level"), ("r", "right_eye_DR_Level")):
                    g = grp[grp["image_id"].astype(str).str.contains(f"_{eye}", regex=False)]
                    def drop(reason, grade=None):
                        excl.append({"subset": sub, "patient_id": int(pid), "eye": eye,
                                     "reason": reason, "n_views": len(g),
                                     "image_ids": ",".join(g["image_id"].astype(str)), "grade": grade})
                    if len(g) == 0: continue
                    if len(g) < 2: drop("MISSING_VIEW"); continue
                    lv = g[col].dropna() if col in g else pd.Series(dtype=float)
                    if lv.empty: drop("MISSING_GRADE"); continue
                    grade = int(lv.iloc[0])
                    if not 0 <= grade < NUM_CLASSES: drop("INVALID_GRADE", grade); continue
                    gg = g.assign(_k=g["image_id"].astype(str).str[-1]).sort_values("_k")
                    ids = gg["image_id"].astype(str).tolist()[:2]
                    paths = [disk.get(i) for i in ids]
                    if any(p is None for p in paths): drop("PATH_NOT_FOUND", grade); continue
                    recs.append({"patient_id": int(pid), "eye": eye, "f1": paths[0], "f2": paths[1],
                                 "grade": grade})
        self.df, self.excl = pd.DataFrame(recs), pd.DataFrame(excl)
        if len(self.excl): print(f"  {'+'.join(subs)}: {len(self.excl)} eye(s) excluded "
                                 f"{self.excl.reason.value_counts().to_dict()}")
    __len__ = SetCDataset.__len__
    __getitem__ = SetCDataset.__getitem__

EXT_ROWS, EXT_PRED = [], {}
if DEEPDRID_ROOT is None:
    gate("Gate11_External", False, "DeepDRiD not present in this runtime", blocking=BLOCK)
else:
    parts = {"setC": SetCDataset(DEEPDRID_ROOT),
             "setB": SetABDataset(DEEPDRID_ROOT, ("regular-fundus-validation",)),
             "setA": SetABDataset(DEEPDRID_ROOT, ("regular-fundus-training",))}
    audit = pd.concat([p.excl.assign(partition=k) for k, p in parts.items() if len(p.excl)],
                      ignore_index=True) if any(len(p.excl) for p in parts.values()) else pd.DataFrame()
    audit.to_csv(f"{MET_DIR}/deepdrid_exclusion_audit.csv", index=False)

    counts = pd.DataFrame([{"partition": k, "eyes": len(p.df),
                            "patients": int(p.df.patient_id.nunique()) if len(p.df) else 0,
                            "images": 2 * len(p.df),
                            "excluded": int((audit.partition == k).sum()) if len(audit) else 0}
                           for k, p in parts.items()])
    counts.to_csv(f"{TAB_DIR}/table_06b_deepdrid_counts.csv", index=False)
    print(counts.to_string(index=False))

    # Set-C is the confirmatory partition, so it must be structurally complete.
    sc = parts["setC"]
    gate("Gate11a_SetC_Completeness",
         len(sc.df) == 200 and (len(sc.df) and int(sc.df.patient_id.nunique()) == 100),
         f"Set-C: {int(sc.df.patient_id.nunique()) if len(sc.df) else 0} patients / {len(sc.df)} eyes "
         f"/ {2*len(sc.df)} images (expected 100 / 200 / 400)", blocking=BLOCK)

    pats = {k: set(p.df.patient_id) if len(p.df) else set() for k, p in parts.items()}
    ov = {f"{a}&{b}": len(pats[a] & pats[b]) for a, b in
          (("setA", "setB"), ("setA", "setC"), ("setB", "setC"))}
    gate("Gate11b_PartitionsDisjoint", sum(ov.values()) == 0,
         f"patients {{k: len(v) for k, v in pats.items()}} -> overlaps {ov}"
         .replace("{k: len(v) for k, v in pats.items()}", str({k: len(v) for k, v in pats.items()})))

    ext_models = [("teacher", "-", TEACHER, "FP32", False),
                  ("best_fp32", BEST_SEED, BEST_FP32_MODEL, "FP32", False)]
    if BEST_CONDITION != "dual_csd":
        ext_models.append(("best_csd_fp32", BEST_CSD_SEED, load_student(BEST_CSD_CKPT), "FP32", False))
    for cond, store in (("ptq_int8", PTQ_M), ("ft_ptq_int8", FTPTQ_M), ("qat_int8", QAT_M)):
        sd = REP_SEED.get(cond)
        if sd in store: ext_models.append((cond, sd, store[sd], QUANT_OF[cond], True))

    for pname, ds in parts.items():
        if len(ds.df) == 0: continue
        for order in DEEPDRID_ORDERS:
            ds.order = order
            ld = loader(ds, 16, False)
            primary = (pname == DEEPDRID_PRIMARY_SUBSET and order == DEEPDRID_PRIMARY_ORDER)
            print(f"\nDeepDRiD [{pname} | {order}]: {len(ds.df)} eyes, "
                  f"{ds.df.patient_id.nunique()} patients"
                  + ("   <-- PRE-REGISTERED PRIMARY (confirmatory)" if primary else ""))
            for cond, seed, mdl, quant, on_cpu in ext_models:
                if on_cpu:
                    r = predict_cpu(mdl, ld)
                    yt, yp, pc, cid = r["y_true"], r["y_pred"], r["p_dual"], r["cluster_ids"]
                else:
                    yt, yp, pc, cid = predict(mdl, ld, clusters=True)
                m = all_metrics(yt, yp, pc)
                EXT_ROWS.append({"condition": cond, "seed": seed, "quantization": quant,
                                 "subset": pname, "field_order": order,
                                 "role": "PRIMARY_CONFIRMATORY" if primary else "supplementary",
                                 "eyes": len(ds.df), "patients": int(ds.df.patient_id.nunique()), **m})
                save_predictions(cond, seed, yt, yp, pc, cid, quant, dataset="DeepDRiD",
                                 split=f"{pname}_{order.replace('=', '')}")
                if primary: EXT_PRED[(cond, seed, quant)] = {"y_true": yt, "y_pred": yp,
                                                             "cluster_ids": cid}
                print(f"  {cond:14s} QWK={m['QWK']:.4f} Acc={m['Accuracy']:.4f} "
                      f"MacroF1={m['MacroF1']:.4f}")

    EXT = pd.DataFrame(EXT_ROWS)
    EXT.to_csv(f"{TAB_DIR}/table_06_external_deepdrid.csv", index=False)
    gate("Gate11_External", len(EXT) > 0,
         f"{EXT.condition.nunique()} models x {EXT.subset.nunique()} partitions x "
         f"{EXT.field_order.nunique()} field orders; PRIMARY = {DEEPDRID_PRIMARY_SUBSET}/"
         f"{DEEPDRID_PRIMARY_ORDER}", blocking=BLOCK)

EXT_DF = pd.DataFrame(EXT_ROWS) if EXT_ROWS else pd.DataFrame()

In [ ]:
# ---- patient-clustered bootstrap on the confirmatory partition ----
# A patient contributes two eyes that are not independent, so PATIENTS are resampled, not eyes.
def patient_bootstrap(yt, yp, cl, B=None, rng_seed=11):
    B = B or BOOTSTRAP_B
    fns = metric_fns(); uniq = np.unique(cl)
    idx = {c: np.where(cl == c)[0] for c in uniq}
    rng = np.random.default_rng(rng_seed); draws = {k: np.empty(B) for k in fns}
    for b in range(B):
        pick = np.concatenate([idx[c] for c in rng.choice(uniq, len(uniq), replace=True)])
        for k, fn in fns.items(): draws[k][b] = fn(yt[pick], yp[pick])
    return {k: {"point": float(fns[k](yt, yp)),
                "ci_low": float(np.percentile(d, 2.5)), "ci_high": float(np.percentile(d, 97.5))}
            for k, d in draws.items()}, len(uniq)

def paired_patient_bootstrap(a, b, B=None, rng_seed=13):
    B = B or BOOTSTRAP_B
    ka = next((k for k in EXT_PRED if k[0] == a), None)
    kb = next((k for k in EXT_PRED if k[0] == b), None)
    if ka is None or kb is None: return None
    ra, rb = EXT_PRED[ka], EXT_PRED[kb]
    yt, cl = ra["y_true"], ra["cluster_ids"]
    fns = metric_fns(); uniq = np.unique(cl); idx = {c: np.where(cl == c)[0] for c in uniq}
    rng = np.random.default_rng(rng_seed); draws = {k: np.empty(B) for k in fns}
    for i in range(B):
        pick = np.concatenate([idx[c] for c in rng.choice(uniq, len(uniq), replace=True)])
        for k, fn in fns.items():
            draws[k][i] = fn(yt[pick], ra["y_pred"][pick]) - fn(yt[pick], rb["y_pred"][pick])
    out = []
    for k, d in draws.items():
        lo, hi = np.percentile(d, [2.5, 97.5])
        out.append({"comparison": f"{a}_vs_{b}", "metric": k,
                    "observed_diff": float(fns[k](yt, ra["y_pred"]) - fns[k](yt, rb["y_pred"])),
                    "ci_low": float(lo), "ci_high": float(hi),
                    "excludes_zero": bool(lo > 0 or hi < 0)})
    return out

EXT_CI, EXT_PAIRED = [], []
if EXT_PRED:
    print(f"Patient-clustered bootstrap on {DEEPDRID_PRIMARY_SUBSET} (B={BOOTSTRAP_B}):")
    for (cond, seed, quant), r in EXT_PRED.items():
        res, npat = patient_bootstrap(r["y_true"], r["y_pred"], r["cluster_ids"])
        for met, v in res.items():
            EXT_CI.append({"condition": cond, "seed": seed, "quantization": quant,
                           "subset": DEEPDRID_PRIMARY_SUBSET, "n_patients": npat, "metric": met, **v})
        q = res["QWK"]
        print(f"  {cond:14s} QWK={q['point']:.4f} [{q['ci_low']:.4f}, {q['ci_high']:.4f}] "
              f"({npat} patients)")
    for a, b in (("ptq_int8", "best_fp32"), ("qat_int8", "best_fp32"),
                 ("ft_ptq_int8", "best_fp32"), ("qat_int8", "ptq_int8")):
        r = paired_patient_bootstrap(a, b)
        if r:
            EXT_PAIRED += r
            q = next(x for x in r if x["metric"] == "QWK")
            print(f"  {a} vs {b}: dQWK={q['observed_diff']:+.4f} "
                  f"[{q['ci_low']:+.4f}, {q['ci_high']:+.4f}] credible={q['excludes_zero']}")
    if EXT_CI: pd.DataFrame(EXT_CI).to_csv(f"{TAB_DIR}/table_06c_external_patient_ci.csv", index=False)
    if EXT_PAIRED: pd.DataFrame(EXT_PAIRED).to_csv(f"{TAB_DIR}/table_06d_external_paired.csv", index=False)
gate("Gate11c_ExternalCIs", bool(EXT_CI),
     f"{len(EXT_CI)} patient-clustered intervals, {len(EXT_PAIRED)} paired comparisons")

## 20 — Export & deployment verification

Artifacts: `checkpoint.pt` (a **pure** state_dict), `model.pt2` (`torch.export`), `model.onnx`,
`metadata.json`. Export failures are reported as failures, never downgraded to a pass.

Every model — FP32 and INT8 alike — is **reloaded from disk into a fresh object** and checked for
numeric parity, a valid grade range and stability over repeated inference. INT8 models are rebuilt
by reconstructing the quantized skeleton (fuse → prepare → convert) and loading the saved
`state_dict` into it, which is the supported way to restore a quantized model.

`selected_deployment/` is populated **only after** verification passes.

In [ ]:
class InferenceWrapper(nn.Module):
    def __init__(self, model): super().__init__(); self.model = model
    def forward(self, macula, disc): return self.model(macula, disc)["p_dual"]

_parity_batch = (_sample["macula"].cpu(), _sample["disc"].cpu())

def export_model(model, name, on_cpu=False, meta=None):
    d = f"{MODELS_DIR}/{name}"; os.makedirs(d, exist_ok=True)
    m = (model if on_cpu else copy.deepcopy(model).to("cpu")).eval()
    ex_m, ex_d = _parity_batch[0][:1], _parity_batch[1][:1]
    st = {"state_dict": False, "pt2": False, "onnx": False, "onnx_max_abs_diff": None,
          "onnx_same_grades": None, "error": None}
    try:
        torch_save({k: v.detach().cpu() for k, v in m.state_dict().items()}, f"{d}/checkpoint.pt")
        st["state_dict"] = True
    except Exception as e: st["error"] = f"state_dict: {e!r}"
    wrap = InferenceWrapper(m).eval()
    try:
        torch.export.save(torch.export.export(wrap, (ex_m, ex_d)), f"{d}/model.pt2"); st["pt2"] = True
    except Exception as e:
        st["error"] = f"pt2: {e!r}"; print(f"  [{name}] torch.export failed: {e!r}")
    try:
        torch.onnx.export(wrap, (ex_m, ex_d), f"{d}/model.onnx", input_names=["macula", "disc"],
                          output_names=["p_cumulative"],
                          dynamic_axes={"macula": {0: "batch"}, "disc": {0: "batch"},
                                        "p_cumulative": {0: "batch"}}, dynamo=True)
        st["onnx"] = True
        import onnxruntime as ort
        sess = ort.InferenceSession(f"{d}/model.onnx", providers=["CPUExecutionProvider"])
        pm, pdd = _parity_batch
        with torch.no_grad(): ref = wrap(pm, pdd).numpy()
        got = sess.run(None, {"macula": pm.numpy(), "disc": pdd.numpy()})[0]
        st["onnx_max_abs_diff"] = float(np.max(np.abs(got - ref)))
        st["onnx_same_grades"] = bool(((got > 0.5).sum(1) == (ref > 0.5).sum(1)).all())
        st["onnx_n_samples"] = int(pm.shape[0])
    except Exception as e:
        st["error"] = f"onnx: {e!r}"; print(f"  [{name}] ONNX failed: {e!r}")
    save_json({"model_name": name, "architecture": type(m).__name__, "dataset": "DRTiD",
               "grade_mapping": list(range(NUM_CLASSES)), **PREPROCESSING,
               "torch_version": torch.__version__, "artifacts": st, **(meta or {})},
              f"{d}/metadata.json")
    print(f"  [{name}] state_dict={st['state_dict']} pt2={st['pt2']} onnx={st['onnx']} "
          f"parity={st['onnx_max_abs_diff']}")
    return d, st

def ptq_skeleton():
    """An architecturally identical INT8 model with placeholder qparams; loading the saved
    state_dict restores the real packed weights, scales and zero-points."""
    m = fresh_student(BEST_CKPT); m.eval()
    m.backbone.fuse_model(qat=False)
    m.backbone = QuantizableBackbone(m.backbone)
    m.backbone.qconfig = get_default_qconfig(QUANT_ENGINE)
    prep = prepare(m, inplace=False)
    with torch.no_grad():
        for i, b in enumerate(calib_loader()):
            prep(b["macula"], b["disc"])
            if i >= 0: break                       # observers only need to see data before convert
    return convert(prep, inplace=False)

def qat_skeleton():
    return convert(build_qat_prepared(BEST_CKPT).to("cpu").eval(), inplace=False)

def verify_deployment(d, builder, reference, n_runs=100):
    """Proves the SAVED ARTIFACT works -- not the object still alive in RAM."""
    checks = {}
    ex_m, ex_d = _parity_batch[0][:1], _parity_batch[1][:1]
    ck = f"{d}/checkpoint.pt"
    checks["checkpoint_exists"] = os.path.exists(ck)
    with torch.no_grad():
        o = reference(ex_m, ex_d)
        ref = (o["p_dual"] if isinstance(o, dict) else o).cpu()
    try:
        fresh = builder()
        fresh.load_state_dict(load_weights(ck, "cpu")); fresh.eval()
        with torch.no_grad():
            o = fresh(ex_m, ex_d)
            got = (o["p_dual"] if isinstance(o, dict) else o).cpu()
        checks["reloaded_from_disk"] = True
        g = int((got > 0.5).sum())
        checks["grade_in_range"] = bool(0 <= g <= NUM_CLASSES - 1)
        checks["max_abs_diff"] = float(torch.max(torch.abs(got - ref)))
        checks["parity_ok"] = checks["max_abs_diff"] < 1e-5
        with torch.no_grad():
            for _ in range(n_runs): fresh(ex_m, ex_d)
        checks["stable_over_runs"] = True
    except Exception as e:
        checks["reloaded_from_disk"] = False; checks["error"] = repr(e)
    checks["deployment_verified"] = bool(checks.get("reloaded_from_disk") and
                                         checks.get("parity_ok") and checks.get("grade_in_range")
                                         and checks.get("stable_over_runs"))
    return checks

EXPORTS, DEPLOY_CHECKS = {}, {}
EXPORTS["teacher_fp32"] = export_model(TEACHER, "teacher_fp32", meta={"role": "upper_bound_teacher"})
EXPORTS["best_student_fp32"] = export_model(
    BEST_FP32_MODEL, "best_student_fp32",
    meta={"role": "deployment_candidate", "condition": BEST_CONDITION, "seed": BEST_SEED})
if BEST_CONDITION != "dual_csd":
    EXPORTS["best_csd_fp32"] = export_model(load_student(BEST_CSD_CKPT), "best_csd_fp32",
                                            meta={"role": "best_csd_artifact", "seed": BEST_CSD_SEED})
for cond, store, nm in (("ptq_int8", PTQ_M, "best_student_ptq_int8"),
                        ("ft_ptq_int8", FTPTQ_M, "best_student_ft_ptq_int8"),
                        ("qat_int8", QAT_M, "best_student_qat_int8")):
    sd = REP_SEED.get(cond)
    if sd in store:
        EXPORTS[nm] = export_model(store[sd], nm, on_cpu=True,
                                   meta={"role": "deployment_int8", "quantization": QUANT_OF[cond],
                                         "seed": sd, **{k: v for k, v in
                                                        (PTQ_I if cond == "ptq_int8" else
                                                         FTPTQ_I if cond == "ft_ptq_int8" else
                                                         QAT_I)[sd].items()
                                                        if k != "quantized_module_paths"}})

DEPLOY_CHECKS["best_student_fp32"] = verify_deployment(
    EXPORTS["best_student_fp32"][0], lambda: Student(init=INIT_TH),
    copy.deepcopy(BEST_FP32_MODEL).to("cpu").eval())
for nm, cond, store, builder in (("best_student_ptq_int8", "ptq_int8", PTQ_M, ptq_skeleton),
                                 ("best_student_ft_ptq_int8", "ft_ptq_int8", FTPTQ_M, ptq_skeleton),
                                 ("best_student_qat_int8", "qat_int8", QAT_M, qat_skeleton)):
    sd = REP_SEED.get(cond)
    if nm in EXPORTS and sd in store:
        DEPLOY_CHECKS[nm] = verify_deployment(EXPORTS[nm][0], builder, store[sd])
save_json(DEPLOY_CHECKS, f"{RESULTS_DIR}/deployment_verification.json")
print(json.dumps(DEPLOY_CHECKS, indent=2, default=str))

_state_ok = all(s["state_dict"] for _, s in EXPORTS.values())
_fp32_names = [n for n in EXPORTS if "int8" not in n]
_fp32_parity = {n: EXPORTS[n][1].get("onnx_max_abs_diff") for n in _fp32_names}
_fp32_ok = all(EXPORTS[n][1]["onnx"] and EXPORTS[n][1].get("onnx_same_grades")
               and (EXPORTS[n][1].get("onnx_max_abs_diff") or 1) < 1e-3 for n in _fp32_names)
gate("Gate12a_Artifacts", _state_ok, f"{len(EXPORTS)} models, every state_dict written",
     blocking=BLOCK)
# FP32 ONNX only: eager INT8 carries packed params that torch.export cannot trace, and that is not
# a deployment blocker because the INT8 path is skeleton + state_dict.
gate("Gate12b_FP32_ONNX", _fp32_ok,
     f"FP32 models {_fp32_names}; ONNX Runtime max|diff| {_fp32_parity}; identical grades",
     blocking=BLOCK)
gate("Gate12c_ArtifactReload",
     bool(DEPLOY_CHECKS) and all(v.get("deployment_verified") for v in DEPLOY_CHECKS.values()),
     "; ".join(f"{k}={'verified' if v.get('deployment_verified') else 'NOT verified'}"
               for k, v in DEPLOY_CHECKS.items()), blocking=BLOCK)

In [ ]:
# ---- selected_deployment/: populated ONLY after verification ----
import shutil
DEPLOY_EXPORT = {"best_fp32": "best_student_fp32", "ptq_int8": "best_student_ptq_int8",
                 "ft_ptq_int8": "best_student_ft_ptq_int8",
                 "qat_int8": "best_student_qat_int8"}.get(DEPLOY_CHOICE, "best_student_fp32")
_verified = bool(DEPLOY_CHECKS.get(DEPLOY_EXPORT, {}).get("deployment_verified"))
SEL_DIR = f"{MODELS_DIR}/selected_deployment"; os.makedirs(SEL_DIR, exist_ok=True)
copied = []
if _verified and DEPLOY_EXPORT in EXPORTS:
    for fn in os.listdir(EXPORTS[DEPLOY_EXPORT][0]):
        shutil.copy2(os.path.join(EXPORTS[DEPLOY_EXPORT][0], fn), os.path.join(SEL_DIR, fn))
        copied.append(fn)
else:
    print(f"  !! {DEPLOY_EXPORT} did not pass verification -- selected_deployment/ not published")

def metrics_of(df, cond, seed=None, cols=("QWK", "MacroF1", "Accuracy", "MAE", "SevereErrorRate")):
    if df is None or not len(df) or cond not in set(df.get("condition", [])): return {}
    s = df[df.condition == cond]
    if seed is not None and "seed" in s and (s.seed == seed).any(): s = s[s.seed == seed]
    return {c: float(s[c].mean()) for c in cols if c in s.columns}

_ext_primary = EXT_DF[EXT_DF.role == "PRIMARY_CONFIRMATORY"] if len(EXT_DF) else pd.DataFrame()
save_json({"deployment_choice": DEPLOY_CHOICE, "seed": DEPLOY_SEED, "quantization": DEPLOY_QUANT,
           "method": BEST_CONDITION, "source_model": DEPLOY_EXPORT, "verified": _verified,
           "selection_rule": DEPLOY_RULE, "selection_reason": DEPLOY_REASON,
           "selected_on": "DRTiD validation only, frozen before test/external evaluation",
           "drtid_validation": RQ2_VAL_SUM[RQ2_VAL_SUM.condition == DEPLOY_CHOICE].to_dict("records"),
           "drtid_test": metrics_of(RAW, DEPLOY_CHOICE, DEPLOY_SEED),
           "deepdrid_setC": metrics_of(_ext_primary, DEPLOY_CHOICE),
           "grade_mapping": {0: "No DR", 1: "Mild NPDR", 2: "Moderate NPDR",
                             3: "Severe NPDR", 4: "Proliferative DR"},
           "preprocessing": PREPROCESSING, "quantization_scope": QUANT_SCOPE,
           "quantization_engine": QUANT_ENGINE, "torch_version": torch.__version__,
           "artifacts": copied,
           "artifact_sha256": {f: sha256_file(os.path.join(SEL_DIR, f)) for f in copied
                               if os.path.isfile(os.path.join(SEL_DIR, f))},
           "disclaimer": "Research prototype; not a standalone clinical diagnosis."},
          f"{SEL_DIR}/metadata.json")
print(f"selected_deployment/ <- {DEPLOY_EXPORT} ({DEPLOY_CHOICE}, seed {DEPLOY_SEED}); files: {copied}")

In [ ]:
# ---- inference interface: serves the SELECTED model, loaded from its artifact ----
GRADE_NAMES = {0: "No DR", 1: "Mild NPDR", 2: "Moderate NPDR", 3: "Severe NPDR", 4: "Proliferative DR"}
DISCLAIMER = ("Research prototype for the DR-VERGE study. Not a medical device and not a standalone "
              "clinical diagnosis; outputs must be reviewed by a clinician.")
_cache = {}

def get_deployment_model():
    if "m" not in _cache:
        ck = f"{SEL_DIR}/checkpoint.pt"
        builder = (qat_skeleton if DEPLOY_CHOICE == "qat_int8" else
                   ptq_skeleton if DEPLOY_CHOICE in ("ptq_int8", "ft_ptq_int8") else
                   (lambda: Student(init=INIT_TH)))
        if not os.path.exists(ck):
            raise RuntimeError(f"selected deployment artifact missing: {ck}")
        m = builder(); m.load_state_dict(load_weights(ck, "cpu")); m.eval()
        _cache["m"] = m
        print(f"  deployment model '{DEPLOY_CHOICE}' loaded from {ck}")
    return _cache["m"]

def predict_dr(macula_image, disc_image):
    """macula_image / disc_image: HxWx3 uint8 RGB arrays or file paths."""
    m = get_deployment_model()
    prep = lambda x: eval_tf(image=(np.array(Image.open(x).convert("RGB")) if isinstance(x, str)
                                    else np.asarray(x)))["image"].unsqueeze(0)
    a, b = prep(macula_image), prep(disc_image)
    t0 = time.perf_counter()
    with torch.no_grad():
        o = m(a, b); p = (o["p_dual"] if isinstance(o, dict) else o)[0]
    dt = (time.perf_counter() - t0) * 1000
    cum = p.cpu().numpy(); grade = int((cum > 0.5).sum())
    ext = np.concatenate([[1.0], cum, [0.0]])
    scores = np.clip(ext[:-1] - ext[1:], 0, None); scores = scores / max(scores.sum(), 1e-9)
    return {"grade": grade, "grade_name": GRADE_NAMES[grade],
            "ordinal_scores": cum.tolist(), "grade_scores": scores.tolist(),
            # Deliberately not called a probability: weighted-BCE training distorts these sigmoids.
            "uncalibrated_score": float(scores[grade]),
            "model_version": f"{DEPLOY_CHOICE}_seed{DEPLOY_SEED}_{DEPLOY_QUANT}",
            "quantization": DEPLOY_QUANT, "latency_ms": dt, "disclaimer": DISCLAIMER}

_demo = pd.read_csv(TEST_CSV).iloc[0]
_out = predict_dr(f"{DRTID_IMAGE_ROOT}/{_demo['macula_filename']}",
                  f"{DRTID_IMAGE_ROOT}/{_demo['disc_filename']}")
print("predict_dr() demo ->", json.dumps({k: v for k, v in _out.items()
                                          if k not in ("ordinal_scores", "grade_scores")},
                                         indent=2, default=str))
print("true grade:", int(_demo["grade"]))

## 21 — Figures & tables

Every figure is written as PNG (400 dpi) + PDF + SVG **and** a companion CSV holding exactly the
numbers plotted, so no value exists only inside an image.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 400, "font.size": 11,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True})

LABELS = {"teacher": "Teacher (ResNet-50 dual-view)", "macula_only": "Student, macula only",
          "disc_only": "Student, optic-disc only", "dual_no_distill": "Dual-view, no distillation",
          "dual_logitkd": "Logit-KD", "dual_featkd": "Logit-KD + Feature-KD",
          "dual_csd": "Logit-KD + CSD (proposed)",
          "abl_csd_raw_smoothl1": "CSD ablation: unscaled Huber",
          "abl_csd_kl_softmax": "CSD ablation: KL-softmax (negative control)",
          "abl_csd_counterfactual": "CSD ablation: same-head counterfactual",
          "best_fp32": "M* (FP32)", "fp32_ft_control": "FP32 fine-tune control (matched graph)",
          "fp32_ft_plain": "FP32 fine-tune (plain)", "ptq_int8": "PTQ INT8",
          "ft_ptq_int8": "FP32 fine-tune -> PTQ INT8", "qat_int8": "QAT INT8",
          "best_csd_fp32": "Best CSD (FP32)"}
def pretty(c): return LABELS.get(c, c)
ORDER = ["teacher", "macula_only", "disc_only", "dual_no_distill", "dual_logitkd", "dual_featkd",
         "dual_csd", "abl_csd_raw_smoothl1", "abl_csd_kl_softmax", "abl_csd_counterfactual",
         "best_fp32", "fp32_ft_control", "fp32_ft_plain", "ptq_int8", "ft_ptq_int8", "qat_int8"]
pd.DataFrame([{"condition": k, "paper_label": v} for k, v in LABELS.items()]).to_csv(
    f"{TAB_DIR}/table_condition_labels.csv", index=False)

def save_fig(fig, stem, data, caption=""):
    for ext in ("png", "pdf", "svg"): fig.savefig(f"{FIG_DIR}/{stem}.{ext}", bbox_inches="tight")
    data.to_csv(f"{FIG_DIR}/{stem}_data.csv", index=False)
    if caption:
        with open(f"{FIG_DIR}/{stem}_caption.txt", "w") as f: f.write(caption)
    plt.close(fig); print(f"  saved {stem}")

def present(order, df=None):
    df = RAW if df is None else df
    return [c for c in order if c in set(df["condition"])]

def agg_stat(df, conds, col):
    mu, sd = [], []
    for c in conds:
        v = df[df.condition == c][col].dropna()
        mu.append(v.mean() if len(v) else np.nan); sd.append(v.std() if len(v) > 1 else 0.0)
    return np.array(mu), np.array(sd)

# Figure 1 -- dataset composition
fig, ax = plt.subplots(figsize=(8, 5))
st = pd.read_csv(f"{TAB_DIR}/table_00_dataset_statistics.csv")
bot = np.zeros(len(st))
rows1 = []
for g in range(NUM_CLASSES):
    v = st[f"grade_{g}"].values
    ax.bar(st["split"], v, bottom=bot, label=f"Grade {g}"); bot += v
    rows1 += [{"split": s, "grade": g, "n": int(n)} for s, n in zip(st["split"], v)]
ax.set_ylabel("eyes"); ax.set_title("Figure 1 - DRTiD split composition"); ax.legend(fontsize=8)
save_fig(fig, "fig_01_dataset", pd.DataFrame(rows1),
         "DRTiD eye counts per split and grade. Grade 4 is ~3.9% of training, which is why the "
         "train/val split is stratified.")

# Figure 2 -- predictive performance across the main conditions
conds = present(["teacher", "macula_only", "disc_only"] + CORE_CONDITIONS)
mets = ["QWK", "MacroF1", "Accuracy"]
rows2 = []
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, met in zip(axes, mets):
    mu, sd = agg_stat(RAW, conds, met)
    ax.bar(range(len(conds)), mu, yerr=sd, capsize=4, color="#4C72B0", edgecolor="#25405e")
    ax.set_xticks(range(len(conds)))
    ax.set_xticklabels([pretty(c) for c in conds], rotation=35, ha="right", fontsize=8)
    ax.set_title(f"{met} (higher is better)")
    rows2 += [{"metric": met, "condition": c, "mean": m_, "sd": s_} for c, m_, s_ in zip(conds, mu, sd)]
fig.suptitle("Figure 2 - Predictive performance on the DRTiD test set", y=1.02)
save_fig(fig, "fig_02_performance", pd.DataFrame(rows2),
         "Mean +/- SD across seeds on the internal test set.")

# Figure 3 -- ordinal safety
rows3 = []
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, met in zip(axes, ["MAE", "SevereErrorRate"]):
    mu, sd = agg_stat(RAW, conds, met)
    ax.bar(range(len(conds)), mu, yerr=sd, capsize=4, color="#C44E52", edgecolor="#5e2540")
    ax.set_xticks(range(len(conds)))
    ax.set_xticklabels([pretty(c) for c in conds], rotation=35, ha="right", fontsize=8)
    ax.set_title(f"{met} (lower is better)")
    rows3 += [{"metric": met, "condition": c, "mean": m_, "sd": s_} for c, m_, s_ in zip(conds, mu, sd)]
fig.suptitle("Figure 3 - Ordinal safety: MAE and severe-error rate", y=1.02)
save_fig(fig, "fig_03_ordinal_safety", pd.DataFrame(rows3),
         "Severe error = |predicted - true| >= 2 grades, the clinically consequential mistake.")

# Figure 4 -- per-grade recall
rows4 = []
fig, ax = plt.subplots(figsize=(10, 5.5))
x = np.arange(NUM_CLASSES); w = 0.8 / max(len(conds), 1)
for i, c in enumerate(conds):
    v = [RAW[RAW.condition == c][f"Recall_Grade{g}"].mean() for g in range(NUM_CLASSES)]
    ax.bar(x + i * w - 0.4, v, w, label=pretty(c))
    rows4 += [{"condition": c, "grade": g, "recall": r} for g, r in zip(range(NUM_CLASSES), v)]
ax.set_xticks(x); ax.set_xticklabels([f"Grade {g}" for g in range(NUM_CLASSES)])
ax.set_ylabel("recall"); ax.legend(fontsize=7, ncol=2)
ax.set_title("Figure 4 - Per-grade recall (intermediate grades are the hard ones)")
save_fig(fig, "fig_04_per_grade_recall", pd.DataFrame(rows4),
         "Per-grade recall. Rare intermediate grades are where ordinal models usually fail.")

In [ ]:
# Figure 5 -- confusion matrices for the core conditions
cc = present(CORE_CONDITIONS)
rows5 = []
fig, axes = plt.subplots(1, len(cc), figsize=(4.2 * len(cc), 4))
axes = np.atleast_1d(axes)
for ax, c in zip(axes, cc):
    s = RAW[RAW.condition == c]
    seed = int(s.iloc[0]["seed"])
    f = f"{MET_DIR}/confusion_{c}_seed{seed}_FP32.csv"
    if not os.path.exists(f): ax.axis("off"); continue
    cm = pd.read_csv(f, index_col=0).values
    cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    ax.set_title(pretty(c), fontsize=9)
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if cmn[i, j] > 0.5 else "black")
            rows5.append({"condition": c, "true": i, "pred": j, "count": int(cm[i, j]),
                          "normalized": float(cmn[i, j])})
fig.suptitle("Figure 5 - Row-normalized confusion matrices (diagonal = per-grade recall)", y=1.04)
save_fig(fig, "fig_05_confusion", pd.DataFrame(rows5), "Row-normalized confusion matrices.")

# Figure 6 -- dual-view gain
rows6 = []
dv = present(CORE_CONDITIONS + ["best_fp32"])
fig, ax = plt.subplots(figsize=(9, 5))
for i, col in enumerate(["DualViewGain_G_aux", "DualViewGain_G_independent"]):
    mu, sd = agg_stat(RAW, dv, col)
    ax.bar(np.arange(len(dv)) + i * 0.4 - 0.2, mu, 0.4, yerr=sd, capsize=3,
           label="G_aux (own auxiliary heads)" if i == 0 else "G_independent (separate models)")
    rows6 += [{"metric": col, "condition": c, "mean": m_} for c, m_ in zip(dv, mu)]
ax.axhline(0, color="#333", lw=1); ax.set_xticks(range(len(dv)))
ax.set_xticklabels([pretty(c) for c in dv], rotation=30, ha="right", fontsize=8)
ax.set_ylabel("QWK gain"); ax.legend(fontsize=8)
ax.set_title("Figure 6 - Dual-view gain, measured two ways")
save_fig(fig, "fig_06_dual_view_gain", pd.DataFrame(rows6),
         "G_aux compares the dual head against the model's own auxiliary heads; G_independent "
         "against independently trained single-view students on the same seed.")

# Figure 7 -- CSD mechanism
mech = present(CORE_CONDITIONS + ["abl_csd_counterfactual"])
rows7 = []
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, met in zip(axes, ["ShiftL1", "CosAgree", "BenefitCorr"]):
    mu, sd = agg_stat(RAW, mech, met)
    ax.bar(range(len(mech)), mu, yerr=sd, capsize=4, color="#55A868", edgecolor="#2f5d3f")
    ax.set_xticks(range(len(mech)))
    ax.set_xticklabels([pretty(c) for c in mech], rotation=35, ha="right", fontsize=8)
    ax.set_title(f"{met} ({'lower' if met == 'ShiftL1' else 'higher'} is better)")
    rows7 += [{"metric": met, "condition": c, "mean": m_} for c, m_ in zip(mech, mu)]
fig.suptitle("Figure 7 - CSD mechanism: was the teacher's decision shift transferred?", y=1.02)
save_fig(fig, "fig_07_csd_mechanism", pd.DataFrame(rows7),
         "ShiftL1 lower = the student's decision-shift structure is closer to the teacher's; "
         "CosAgree higher = the shift points the same way; BenefitCorr higher = the student gains "
         "from dual-view on the same samples the teacher does.")

# Figure 8 -- CSD gradient contribution over training
rows8 = []
fig, ax = plt.subplots(figsize=(9, 5))
for c in present(CORE_CONDITIONS):
    f = f"{LOG_DIR}/{c}_seed{PRIMARY_SEED}.csv"
    if not os.path.exists(f): continue
    h = pd.read_csv(f)
    if "ratio_csd_over_task" in h:
        ax.plot(h["epoch"], h["ratio_csd_over_task"], lw=1.8, label=pretty(c))
        rows8 += [{"condition": c, "epoch": int(e), "ratio": float(r)}
                  for e, r in zip(h["epoch"], h["ratio_csd_over_task"])]
ax.axhline(1.0, color="#888", ls=":", lw=1)
ax.set_xlabel("epoch"); ax.set_ylabel("||grad L_CSD|| / ||grad L_task||")
ax.set_title("Figure 8 - Is CSD a real training signal?"); ax.legend(fontsize=8)
save_fig(fig, "fig_08_csd_gradient", pd.DataFrame(rows8),
         "Shared-backbone gradient ratio. A term can be numerically present and still contribute "
         "no gradient; this shows whether CSD actually influences learning.")

In [ ]:
# Figure 9 -- quantization retention
qc = present(["best_fp32", "fp32_ft_control", "ptq_int8", "ft_ptq_int8", "qat_int8"])
rows9 = []
if "best_fp32" in qc:
    ref = RAW[RAW.condition == "best_fp32"]
    fig, ax = plt.subplots(figsize=(9, 5))
    for i, met in enumerate(["QWK", "MacroF1", "Accuracy"]):
        r0 = ref[met].mean()
        v = [100 * RAW[RAW.condition == c][met].mean() / r0 for c in qc]
        ax.bar(np.arange(len(qc)) + i * 0.27 - 0.27, v, 0.27, label=met)
        rows9 += [{"metric": met, "condition": c, "retention_pct": x} for c, x in zip(qc, v)]
    ax.axhline(100, color="#333", ls="--", lw=1)
    ax.axhline(DEPLOY_RULE["min_qwk_retention_pct"], color="#C44E52", ls=":", lw=1.4,
               label=f"{DEPLOY_RULE['min_qwk_retention_pct']}% engineering criterion")
    ax.set_xticks(range(len(qc)))
    ax.set_xticklabels([pretty(c) for c in qc], rotation=30, ha="right", fontsize=8)
    ax.set_ylabel("% of FP32"); ax.legend(fontsize=8)
    ax.set_title("Figure 9 - Performance retention after quantization")
    save_fig(fig, "fig_09_retention", pd.DataFrame(rows9),
             "Retention relative to the FP32 model. The 95% line is a pre-specified engineering "
             "criterion for choosing an artifact, not a clinical margin.")

# Figure 10 -- size and latency
rows10 = []
ec = present(["teacher"] + qc)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, col, lbl in ((axes[0], "CheckpointSize_MB", "serialized size (MB)"),
                     (axes[1], "Latency_median_ms", "median CPU latency (ms)")):
    v = [RAW[RAW.condition == c][col].mean() for c in ec]
    ax.bar(range(len(ec)), v, color="#8172B2", edgecolor="#4a4066")
    ax.set_xticks(range(len(ec)))
    ax.set_xticklabels([pretty(c) for c in ec], rotation=30, ha="right", fontsize=8)
    ax.set_ylabel(lbl); ax.set_yscale("log" if col == "CheckpointSize_MB" else "linear")
    rows10 += [{"metric": col, "condition": c, "value": x} for c, x in zip(ec, v)]
fig.suptitle("Figure 10 - Efficiency: model size and CPU latency", y=1.02)
save_fig(fig, "fig_10_efficiency", pd.DataFrame(rows10),
         "Serialized state_dict size and single-thread CPU latency (median of 5 blocks).")

# Figure 11 -- QWK vs latency Pareto
fig, ax = plt.subplots(figsize=(8, 6))
rows11 = []
for c in ec:
    s = RAW[RAW.condition == c]
    x, y = s["Latency_median_ms"].mean(), s["QWK"].mean()
    ax.scatter(x, y, s=90); ax.annotate(pretty(c), (x, y), textcoords="offset points",
                                        xytext=(6, 5), fontsize=8)
    rows11.append({"condition": c, "latency_ms": x, "QWK": y,
                   "size_MB": s["CheckpointSize_MB"].mean()})
ax.set_xlabel("median CPU latency (ms)"); ax.set_ylabel("test QWK")
ax.set_title("Figure 11 - Efficiency-performance Pareto frontier")
save_fig(fig, "fig_11_pareto", pd.DataFrame(rows11),
         "Accuracy against CPU latency. Points nearer the top-left dominate.")

# Figure 12 -- statistical forest plot
if len(STATS):
    f_ = STATS[STATS.metric == "QWK"].sort_values(["RQ", "comparison"])
    if len(f_):
        fig, ax = plt.subplots(figsize=(10, max(4, 0.5 * len(f_) + 2)))
        yp = np.arange(len(f_))[::-1]
        ax.errorbar(f_["observed_diff"], yp,
                    xerr=[f_["observed_diff"] - f_["ci_low"], f_["ci_high"] - f_["observed_diff"]],
                    fmt="o", capsize=4, lw=1.6, ecolor="#666", ls="none")
        ax.axvline(0, color="#333", ls="--", lw=1.2)
        ax.set_yticks(yp)
        ax.set_yticklabels([f"{r.RQ}  {pretty(r.comparison.split('_vs_')[0])} vs "
                            f"{pretty(r.comparison.split('_vs_')[1])}" for r in f_.itertuples()],
                           fontsize=8)
        ax.set_xlabel("observed dQWK (95% cluster-bootstrap CI)")
        ax.set_title("Figure 12 - Pre-registered comparisons: effect size with 95% CI")
        save_fig(fig, "fig_12_forest", f_,
                 "Forest plot of every pre-registered QWK comparison. The point is the OBSERVED "
                 "paired difference; the bar is the bootstrap interval. An interval crossing zero "
                 "is not a claim.")

# Figure 13 -- Set-C external
if len(EXT_DF) and "role" in EXT_DF:
    prim = EXT_DF[EXT_DF.role == "PRIMARY_CONFIRMATORY"]
    if len(prim):
        ecs = [c for c in ORDER + ["best_csd_fp32"] if c in set(prim.condition)]
        rows13 = []
        fig, axes = plt.subplots(1, 4, figsize=(19, 5))
        for ax, met in zip(axes, ["QWK", "MacroF1", "Accuracy", "SevereErrorRate"]):
            v = [float(prim[prim.condition == c][met].mean()) for c in ecs]
            ax.bar(range(len(ecs)), v, color="#6E9BC5", edgecolor="#25405e")
            ax.set_xticks(range(len(ecs)))
            ax.set_xticklabels([pretty(c) for c in ecs], rotation=35, ha="right", fontsize=8)
            ax.set_title(f"{met} ({'lower' if met == 'SevereErrorRate' else 'higher'} is better)")
            rows13 += [{"metric": met, "condition": c, "value": x} for c, x in zip(ecs, v)]
        fig.suptitle(f"Figure 13 - DeepDRiD {DEEPDRID_PRIMARY_SUBSET} confirmatory external "
                     f"evaluation ({DEEPDRID_PRIMARY_ORDER}, frozen)", y=1.03)
        save_fig(fig, "fig_13_external_setc", pd.DataFrame(rows13),
                 "Confirmatory external evaluation on the DeepDRiD partition reserved for final "
                 "evaluation. No tuning, threshold adjustment or selection happened here.")

# Figure 14 -- internal vs external
if len(EXT_DF) and "role" in EXT_DF:
    prim = EXT_DF[EXT_DF.role == "PRIMARY_CONFIRMATORY"]
    cc2 = [c for c in ORDER + ["best_csd_fp32"] if c in set(prim.condition) and c in set(RAW.condition)]
    if cc2:
        rows14 = []
        fig, axes = plt.subplots(1, 3, figsize=(16, 5))
        for ax, met in zip(axes, ["QWK", "MacroF1", "Accuracy"]):
            iv = [RAW[RAW.condition == c][met].mean() for c in cc2]
            ev = [float(prim[prim.condition == c][met].mean()) for c in cc2]
            x = np.arange(len(cc2))
            ax.bar(x - 0.19, iv, 0.38, label="DRTiD (internal)")
            ax.bar(x + 0.19, ev, 0.38, label="DeepDRiD Set-C (external)")
            ax.set_xticks(x); ax.set_xticklabels([pretty(c) for c in cc2], rotation=35,
                                                 ha="right", fontsize=8)
            ax.set_title(met); ax.legend(fontsize=8)
            rows14 += [{"metric": met, "condition": c, "internal": a, "external": b,
                        "delta": b - a} for c, a, b in zip(cc2, iv, ev)]
        fig.suptitle("Figure 14 - Internal vs confirmatory external generalization", y=1.02)
        save_fig(fig, "fig_14_internal_vs_external", pd.DataFrame(rows14),
                 "A drop on Set-C is a domain-shift finding to report, not something to tune away.")
print(f"\nFigures written to {FIG_DIR}")

In [ ]:
# ---- result tables ----
diag = ["QWK", "Accuracy", "BalancedAccuracy", "MacroPrecision", "MacroRecall", "MacroF1",
        "WeightedF1", "MAE", "SevereErrorRate", "OrdinalThreshold_ECE", "OrdinalThreshold_Brier"]
t1 = RAW.groupby("condition")[diag].agg(["mean", "std"]).round(4)
t1.columns = ["_".join(c) for c in t1.columns]
t1 = t1.reindex([c for c in ORDER if c in t1.index])
t1.to_csv(f"{TAB_DIR}/table_predictive_performance.csv")

eff = [c for c in ["ParamCount", "CheckpointSize_MB", "Latency_median_ms", "Latency_mean_ms",
                   "Latency_sd_ms", "Latency_p95_ms", "Latency_p99_ms", "Latency_block_IQR_ms",
                   "Throughput_pairs_per_s", "Throughput_images_per_s",
                   "PeakRSS_during_inference_MB"] if c in RAW.columns]
t2 = RAW.groupby("condition")[eff].mean().round(4)
if "teacher" in t2.index:
    t2["Speedup_vs_teacher"] = (t2.loc["teacher", "Latency_median_ms"] / t2["Latency_median_ms"]).round(2)
    t2["CompressionRatio_vs_teacher"] = (t2.loc["teacher", "CheckpointSize_MB"] / t2["CheckpointSize_MB"]).round(2)
    t2["SizeReduction_vs_teacher_pct"] = ((1 - t2["CheckpointSize_MB"] /
                                           t2.loc["teacher", "CheckpointSize_MB"]) * 100).round(2)
t2 = t2.reindex([c for c in ORDER if c in t2.index])
t2.to_csv(f"{TAB_DIR}/table_efficiency.csv")

# retention (higher-is-better metrics) and error DELTA (lower-is-better metrics)
rows = []
if "best_fp32" in set(RAW.condition):
    ref = RAW[RAW.condition == "best_fp32"]
    for c in [x for x in ORDER if x in set(RAW.condition)]:
        s = RAW[RAW.condition == c]; row = {"model": c}
        for m in ("QWK", "MacroF1", "Accuracy"):
            row[f"{m}_retention_pct"] = round(100 * s[m].mean() / ref[m].mean(), 2)
        for m in ("MAE", "SevereErrorRate"):
            row[f"delta_{m}"] = round(s[m].mean() - ref[m].mean(), 4)
        rows.append(row)
pd.DataFrame(rows).to_csv(f"{TAB_DIR}/table_retention.csv", index=False)

mech_cols = [c for c in ["ShiftL1", "ShiftMAE", "CosAgree", "BenefitCorr", "BenefitCorrSpearman",
                         "CF_ShiftL1", "CF_CosAgree", "DualViewGain_G_aux",
                         "DualViewGain_G_independent"] if c in RAW.columns]
RAW[RAW.view_mode == "dual"].groupby("condition")[mech_cols].agg(["mean", "std"]).round(4).to_csv(
    f"{TAB_DIR}/table_csd_mechanism.csv")

print("Predictive performance:\n", t1[[c for c in t1.columns if c.endswith("_mean")]].to_string())
print("\nEfficiency:\n", t2.to_string())

## 22 — Final report

In [ ]:
def mean_of(cond, col):
    v = RAW[RAW.condition == cond][col].dropna()
    return float(v.mean()) if len(v) else float("nan")

print("=" * 78); print("HEADLINES (computed, NOT significance claims)"); print("=" * 78)
head = []
tq, sq = mean_of("teacher", "QWK"), mean_of("best_fp32", "QWK")
tp, sp = mean_of("teacher", "ParamCount"), mean_of("best_fp32", "ParamCount")
tl, sl = mean_of("teacher", "Latency_median_ms"), mean_of("best_fp32", "Latency_median_ms")
ts, ss = mean_of("teacher", "CheckpointSize_MB"), mean_of("best_fp32", "CheckpointSize_MB")
if np.isfinite(tp) and sp:
    head.append(f"Teacher -> student: {tp/sp:.0f}x fewer parameters, {ts/ss:.1f}x smaller artifact, "
                f"{tl/sl:.1f}x faster on CPU, {100*sq/tq:.1f}% of teacher QWK retained")
for q in ("ptq_int8", "ft_ptq_int8", "qat_int8"):
    if q in set(RAW.condition):
        head.append(f"FP32 -> {pretty(q)}: {100*mean_of(q,'QWK')/sq:.1f}% QWK retained, "
                    f"{sl/mean_of(q,'Latency_median_ms'):.2f}x CPU speed-up, "
                    f"{ss/mean_of(q,'CheckpointSize_MB'):.2f}x smaller")
for h in head: print("  " + h)
pd.DataFrame({"headline": head}).to_csv(f"{TAB_DIR}/table_headlines.csv", index=False)

print()
print("=" * 78); print("RQ1 VERDICT"); print("=" * 78)
verdict = {}
csd_q = mean_of("dual_csd", "QWK")
for base in ("dual_no_distill", "dual_logitkd", "dual_featkd"):
    if base not in set(RAW.condition): continue
    st = STATS[(STATS.comparison == f"dual_csd_vs_{base}") & (STATS.metric == "QWK")]
    ci = f"[{st.iloc[0]['ci_low']:+.4f}, {st.iloc[0]['ci_high']:+.4f}]" if len(st) else "n/a"
    cred = bool(st.iloc[0]["excludes_zero"]) if len(st) else None
    verdict[base] = {"csd_qwk": csd_q, "baseline_qwk": mean_of(base, "QWK"),
                     "diff": csd_q - mean_of(base, "QWK"), "ci_95": ci, "credible": cred}
    print(f"  CSD vs {base:18s}: {csd_q:.4f} vs {mean_of(base,'QWK'):.4f} "
          f"(diff {csd_q-mean_of(base,'QWK'):+.4f}, 95% CI {ci}, credible={cred})")
print("\n  Mechanism (did the shift transfer, independently of QWK?)")
for c in present(CORE_CONDITIONS):
    print(f"    {c:20s} ShiftL1={mean_of(c,'ShiftL1'):.4f}  CosAgree={mean_of(c,'CosAgree'):+.4f}  "
          f"BenefitCorr={mean_of(c,'BenefitCorr'):+.4f}")
save_json(verdict, f"{RESULTS_DIR}/rq1_verdict.json")
print("\n  A null or negative RQ1 result is a valid, reportable finding. Do not re-tune in")
print("  response to this table -- the protocol was fixed before the run.")

In [ ]:
# ---- final gate report ----
gdf = pd.DataFrame([{"gate": k, "passed": v["passed"], "blocking": v["blocking"],
                     "detail": v["detail"]} for k, v in GATES.items()])
gdf.to_csv(f"{TAB_DIR}/table_gate_report.csv", index=False)
print("=" * 78); print("GATE REPORT"); print("=" * 78)
print(gdf.to_string(index=False))
print(f"\n{int(gdf.passed.sum())}/{len(gdf)} gates passed.")
failed = gdf[~gdf.passed]
if len(failed):
    print("\nFAILED gates (report these honestly rather than hiding them):")
    for _, r in failed.iterrows(): print(f"  - {r['gate']}: {r['detail']}")

save_json({"config": CONFIG, "gates": GATES, "headlines": head, "rq1_verdict": verdict,
           "selection": {"method": BEST_CONDITION, "seed": BEST_SEED,
                         "deployment": DEPLOY_CHOICE, "deployment_seed": DEPLOY_SEED},
           "n_evaluated_runs": int(len(RAW)),
           "external": "completed" if len(EXT_DF) else "skipped"},
          f"{RESULTS_DIR}/run_summary.json")

print(f"""
{'='*78}
ARTIFACTS
{'='*78}
  checkpoints : {CKPT_DIR}
  models      : {MODELS_DIR}   (checkpoint.pt / model.pt2 / model.onnx / metadata.json)
  selected    : {MODELS_DIR}/selected_deployment
  figures     : {FIG_DIR}      (png + pdf + svg + *_data.csv per figure)
  tables      : {TAB_DIR}
  metrics     : {MET_DIR}
  predictions : {PRED_DIR}     (per-sample, so any metric can be recomputed)
  logs        : {LOG_DIR}
  summary     : {RESULTS_DIR}/run_summary.json
""")

## Reading order when writing the paper

1. **`table_gate_report.csv`** — did anything fail? Report failures honestly.
2. **RQ1 verdict** (printed above / `rq1_verdict.json`) — both axes, predictive *and* mechanistic.
3. **`table_predictive_performance.csv`** and **`table_05_statistics.csv`** — effect sizes with CIs.
   A difference whose CI includes zero is not a claim.
4. **`table_04_rq2_validation.csv`** + **`deployment_choice.json`** — the deployment decision and the
   *validation* evidence it rests on. The test set played no part in it.
5. **`table_retention.csv`** and **`table_efficiency.csv`** — RQ2.
6. **`table_06_external_deepdrid.csv`** + **`table_06c_external_patient_ci.csv`** — quote the
   **Set-C** rows under `_1=macula` as the confirmatory external result.
7. **`table_condition_labels.csv`** — the exact label for every condition.

### Wording that must not drift

* CSD transfers **an operational proxy of the dual-view ordinal decision shift** — never "pure
  anatomical complementarity".
* Set-C is **confirmatory external evaluation**; Set-B/Set-A are external validation.
* DRTiD statistics are **eye/record-clustered**; DeepDRiD statistics are patient-clustered.
* Threshold outputs are **cumulative ordinal scores**, not calibrated probabilities.
* Student variability is reported **conditional on a fixed teacher checkpoint**.
* 95% retention is a **pre-specified engineering criterion**, not a clinical margin.
* INT8 quantizes the **backbone only**, so the deployed model is mixed precision; the large
  compression story is teacher → student, the INT8 story is latency.

### Limitations to state

Retrospective public datasets; DRTiD provides no usable patient key, so internal clustering is
eye-level; a fixed teacher checkpoint; Δ is an operational proxy; weighted-BCE scores are not
calibrated probabilities; no lesion-level evidence; no prospective deployment; a Colab CPU is not a
clinic device; two-field input still needs two acquisitions; external population and camera shift;
QWK is not clinical utility.